# DACON 236749 V4 trainable recovery

## tl;dr

**Initial state: not run.** This notebook performs the bounded V4 training experiment only after its Drive inputs pass SHA-256 verification. It records an actual result or a stated censoring/failure reason for six LR probes and the five G0/P1/P2/P3/P4 pipeline rows. It never submits to DACON.

## Context & Methods

The music branch is the only newly trainable component. The runner owns role separation, full-encoder gradient/update checks, matched controls, recipe locking, and the rule that a failed or incomplete candidate retains G0. The 7,200-second A100 allocation includes setup; the runner receives a 7,080-second safe deadline and the notebook watchdog releases the runtime.

## Data

Every archive is reconstructed from Drive parts only after each part and its assembled archive match the embedded receipt. Archives are rejected if a member could escape the workspace. `v4_validation` supplies train, dev, and sealed manifests; only the runner may decide whether the sealed panel is opened.

In [ ]:
import csv, hashlib, importlib.metadata, json, os, shutil, subprocess, sys, threading, time, traceback, zipfile
from pathlib import Path
from google.colab import drive, runtime

PINNED_ALLOCATION_START = 1789283305.0
PINNED_RESUME_RUN = None
SESSION_START = time.time()
ALLOCATION_START = float(PINNED_ALLOCATION_START if PINNED_ALLOCATION_START is not None else os.environ.get('V4_ALLOCATION_STARTED_UNIX', SESSION_START))
SAFE_DEADLINE = ALLOCATION_START + 7080
if SAFE_DEADLINE <= SESSION_START:
    raise TimeoutError('V4 original allocation safe deadline already elapsed')
def watchdog_stop():
    try:
        if 'RUN' in globals() and RUN.exists():
            budget_event('watchdog_release_requested')
    finally:
        runtime.unassign()

watchdog = threading.Timer(max(1, SAFE_DEADLINE - time.time()), watchdog_stop)
watchdog.daemon = True
watchdog.start()
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/DACON_236749')
WORK = Path('/content/dacon_trainable_v4')
WORK.mkdir(exist_ok=True)
resume_run = PINNED_RESUME_RUN or os.environ.get('V4_RESUME_RUN')
if resume_run and Path(resume_run).name != resume_run:
    raise ValueError('V4_RESUME_RUN must be a runs_trainable_v4 child directory name')
RUN = DRIVE / 'runs_trainable_v4' / (resume_run or time.strftime('%Y%m%d_%H%M%S', time.gmtime(SESSION_START)))
if resume_run and not RUN.is_dir():
    raise FileNotFoundError(f'Resume run does not exist: {RUN}')
RUN.mkdir(parents=True, exist_ok=bool(resume_run))

def write_json(path, value):
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, sort_keys=True), encoding='utf-8')
    temporary.replace(path)

def sha256(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

def budget_event(event, **extra):
    with (RUN / 'budget_events.jsonl').open('a', encoding='utf-8') as stream:
        stream.write(json.dumps({'event': event, 'unix': time.time(), 'elapsed_seconds': time.time() - SESSION_START, **extra}) + '\n')

write_json(RUN / ('retry_budget.json' if resume_run else 'budget.json'), {
    'start_unix': SESSION_START, 'allocation_start_unix': ALLOCATION_START, 'safe_deadline_unix': SAFE_DEADLINE, 'resumed_run': bool(resume_run),
    'allocation_limit_seconds': 7200, 'safe_shutdown_seconds': 7080,
})

budget_event('notebook_started', resumed_run=bool(resume_run))
print('RESULTS_DIRECTORY', RUN, flush=True)

### Embedded execution source

These cells are the exact local sources compiled by the builder. The run receipt stores their SHA-256 values before any experiment command starts.

In [ ]:
%%writefile /content/dacon_trainable_v4/train_v4.py
#!/usr/bin/env -S uv run --script
# /// script
# requires-python = ">=3.11"
# dependencies = ["numpy", "torch", "torchaudio", "safetensors", "transformers", "imageio-ffmpeg", "soundfile", "librosa", "einops", "timm", "fvcore", "scikit-learn", "demucs", "panns-inference"]
# ///
# How to run: uv run train_v4.py --help
"""Run the fixed, budgeted V4 full-fine-tuning comparison."""

from __future__ import annotations

import argparse
import csv
import gc
import json
import math
import time
from dataclasses import replace
from pathlib import Path

import torch
import soundfile as sf

from v4_batches import music_supervision_rows
from v4_data import load_manifest
from v4_model import load_model, parameter_groups
from v4_selection import (accept_sealed, mark_sealed_opened, select_development,
    write_recipe_lock, write_sealed_inference_receipt, write_weight_lock)
from v4_training_io import atomic_json, budget_event, digest, g0_predictions, read_predictions
from v4_training_refit import refit_and_seal
from v4_training_trials import TrialSpec, profile_update, train_trial, write_predictions

SEED = 20_260_913
STAGE_END_MINUTES = {"profile": 8, "g0": 20, "probes": 35, "main": 60, "temporal": 80,
                     "refit": 95, "sealed": 105, "export": 110, "shutdown": 118}
TRAINING_SOURCES = tuple(Path(name) for name in ("train_v4.py", "v4_model.py",
    "v4_model_components.py", "v4_training.py", "v4_training_data.py",
    "v4_training_trials.py", "v4_training_refit.py", "v4_training_io.py", "v4_batches.py",
    "v4_data.py", "v4_evaluation.py", "v4_selection.py", "graph_features.py",
    "graph_fusion.py", "graph_scoring.py", "v2_scoring.py", "recovery_sonics.py",
    "ast_multitask.py", "v2_data.py"))


def _budget_schedule(now: float, hard: float) -> dict[str, float]:
    """Scale every stage into the actual time left without extending the allocation."""
    if hard <= now:
        raise TimeoutError("V4 deadline already reached")
    remaining = hard - now
    total = 60 * STAGE_END_MINUTES["shutdown"]
    return {stage: now + remaining * (60 * minutes / total)
            for stage, minutes in STAGE_END_MINUTES.items()}


def _effective_deadline(provided: float, budget: dict, allocation_start: float) -> float:
    safe_seconds = float(budget.get("safe_shutdown_seconds", 7080))
    declared = float(budget.get("safe_deadline_unix", provided))
    return min(provided, declared, allocation_start + safe_seconds)


def _write_epoch_artifacts(output: Path, outcomes: dict[str, dict]) -> None:
    fields = ("pipeline", "epoch", "optimizer_steps", "rank_score", "status")
    with (output / "epoch_metrics.csv").open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        for name, outcome in outcomes.items():
            for report in outcome.get("reports", []):
                writer.writerow({"pipeline": name, "epoch": report["epoch"],
                    "optimizer_steps": report["optimizer_steps"], "rank_score": report["rank_score"],
                    "status": outcome["status"]})
    atomic_json(output / "exposures.json", {name: {
        "cumulative_counts": (value.get("reports") or [{}])[-1].get("cumulative_exposures", {}),
        "records": value.get("exposure_records_path", "")} for name, value in outcomes.items()})


def run(args: argparse.Namespace) -> dict:
    output: Path = args.output
    output.mkdir(parents=True, exist_ok=True)
    budget_path = output / "budget_events.jsonl"
    runner_start = time.time()
    allocation_start = runner_start
    budget = {}
    budget_file = output / "budget.json"
    if budget_file.is_file():
        budget = json.loads(budget_file.read_text())
        allocation_start = float(budget["allocation_start_unix"])
    args.deadline = _effective_deadline(args.deadline, budget, allocation_start)
    stages = _budget_schedule(runner_start, args.deadline)
    atomic_json(output / "stage_budget_plan.json", {"allocation_start_unix": allocation_start,
        "runner_start_unix": runner_start, "setup_elapsed_seconds": max(0, runner_start - allocation_start),
        "hard_deadline_unix": args.deadline, "runner_seconds_remaining": args.deadline - runner_start,
        "stages": {name: {"canonical_ceiling_minutes": STAGE_END_MINUTES[name],
                           "deadline_unix": deadline,
                           "seconds_after_runner_start": deadline - runner_start}
                   for name, deadline in stages.items()}})
    budget_event(budget_path, "runner_started", deadline=args.deadline,
                 setup_elapsed_seconds=max(0, runner_start - allocation_start))
    train_rows = load_manifest(args.data, "train")
    dev_rows = load_manifest(args.data, "dev")
    train = music_supervision_rows(train_rows)
    if not train:
        raise ValueError("train.csv has no justified music supervision")
    eligible_ids = {row["ID"] for row in train}
    eligibility = [{"ID": row["ID"], "eligible_music_supervision": row["ID"] in eligible_ids,
                    "reason": "known_music_present_and_authenticity" if row["ID"] in eligible_ids
                    else "not_a_known_music_present_authenticity_target"} for row in train_rows]
    atomic_json(output / "training_eligibility.json", {"manifest_rows": len(train_rows),
                "eligible_rows": len(train), "excluded_rows": len(train_rows) - len(train),
                "rows": eligibility})
    durations = sorted(((sf.info(args.data / row["path"]).duration, row) for row in dev_rows),
                       key=lambda item: item[0])
    p95_row = durations[max(0, math.ceil(0.95 * len(durations)) - 1)][1]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    assets = {"sonics": args.sonics, "ast": args.ast}
    profiles = []
    for kind in ("sonics", "ast"):
        loaded = load_model(kind, assets[kind], device)
        groups = parameter_groups(loaded)
        profiles.append(profile_update(loaded, (groups.encoder, groups.head), train_rows, args.data,
                                       p95_row, output / "profile_codec",
                                       stages["profile"]))
        del loaded
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
    atomic_json(output / "throughput_profile.json", profiles)
    budget_event(budget_path, "profile_complete", p95_file=p95_row["ID"])
    g0 = g0_predictions(dev_rows, args.data, args.baseline, output / "dev",
                        stages["g0"])
    budget_event(budget_path, "g0_complete", files=len(g0))
    by_kind = {str(profile["kind"]): profile for profile in profiles}
    steps = math.ceil(len(train) / 32)
    probe_seconds = sum(3 * (60 * float(by_kind[kind]["update_seconds"]) * 16
        + len(dev_rows) * float(by_kind[kind]["full_file_inference_seconds"]))
        for kind in ("sonics", "ast"))
    minimum_main_seconds = 5 * steps * 16 * sum(float(by_kind[kind]["update_seconds"])
        for kind in ("sonics", "ast", "sonics")) + 15 * len(dev_rows) * max(
        float(profile["full_file_inference_seconds"]) for profile in profiles)
    atomic_json(output / "pre_search_forecast.json", {"probe_seconds": probe_seconds,
        "minimum_main_seconds": minimum_main_seconds,
        "projected_search_seconds": probe_seconds + minimum_main_seconds,
        "seconds_until_temporal_ceiling": max(0.0, stages["temporal"] - time.time()),
        "fits_before_temporal_ceiling": probe_seconds + minimum_main_seconds <= max(
            0.0, stages["temporal"] - time.time()),
        "policy": "fixed full validation and layer scope; hard stage boundaries censor incomplete trials"})
    outcomes: dict[str, dict] = {}

    def execute(spec: TrialSpec, deadline: float, state: Path | None = None) -> dict:
        model_asset = assets[spec.kind] / ("pytorch_model.bin" if spec.kind == "sonics" else "model.safetensors")
        contract_files = [args.data / "train.csv", args.data / "dev.csv",
                          output / "dev" / "G0_predictions.csv", model_asset, *TRAINING_SOURCES]
        contract_hash = "|".join(digest(path) for path in contract_files)
        spec = replace(spec, contract_hash=contract_hash)
        loaded = load_model(spec.kind, assets[spec.kind], device, state_path=state)
        groups = parameter_groups(loaded)
        try:
            result = train_trial(loaded, (groups.encoder, groups.head), spec, train_rows, dev_rows,
                                 g0, args.data, output / "training" / spec.name, deadline)
        finally:
            del loaded
            gc.collect()
            if device.type == "cuda":
                torch.cuda.empty_cache()
        outcomes[spec.name] = result
        return result

    probe_end = stages["probes"]
    probe_scores: dict[str, list[tuple[float, float]]] = {"sonics": [], "ast": []}
    for kind, rates in (("sonics", (1e-5, 3e-5, 1e-4)), ("ast", (1e-6, 3e-6, 1e-5))):
        for rate in rates:
            name = f"probe_{kind}_{rate:.0e}"
            result = execute(TrialSpec(name, kind, rate, 3 * rate if kind == "sonics" else 1e-4,
                                       epochs=30, max_updates=60, patience=0), probe_end)
            if (result["status"] == "completed" and int(result["updates"]) == 60
                    and math.isfinite(float(result["best_score"]))):
                probe_scores[kind].append((float(result["best_score"]), rate))
    chosen = {kind: max(values, key=lambda item: (item[0], -item[1]))[1]
              for kind, values in probe_scores.items() if values}
    budget_event(budget_path, "probes_complete", completed=sum(map(len, probe_scores.values())))
    main_start, main_end = time.time(), stages["main"]
    main: dict[str, dict] = {}
    p3_epoch_seconds = steps * 16 * (float(by_kind["sonics"]["update_seconds"])
        + float(by_kind["sonics"]["p3_preparation_seconds"]))
    p3_slot_seconds = max(0.0, main_end - main_start) / 3
    p1_epoch_cap = min(30, max(5, math.floor(p3_slot_seconds / max(1e-9, 1.25 * p3_epoch_seconds))))
    atomic_json(output / "augmentation_budget.json", {"p3_profiled_epoch_seconds": p3_epoch_seconds,
        "reserved_seconds": p3_slot_seconds, "safety_factor": 1.25,
        "p1_planned_epoch_cap": p1_epoch_cap,
        "five_epoch_pair_forecast_fits": 5 * 1.25 * p3_epoch_seconds <= p3_slot_seconds})
    if "sonics" in chosen:
        main["P1"] = execute(TrialSpec("P1", "sonics", chosen["sonics"], 3 * chosen["sonics"],
                                         p1_epoch_cap),
                               main_start + max(0, main_end - main_start) / 3)
    if "ast" in chosen:
        main["P2"] = execute(TrialSpec("P2", "ast", chosen["ast"], 1e-4, 30),
                               main_start + 2 * max(0, main_end - main_start) / 3)
    if main.get("P1", {}).get("status") == "completed":
        epochs = int(main["P1"]["epochs_completed"])
        main["P3"] = execute(TrialSpec("P3", "sonics", chosen["sonics"], 3 * chosen["sonics"],
                                         epochs, p3=True, patience=0), main_end)
        if (main["P3"].get("status") == "completed" and
                (main["P3"]["run_total_updates"] != main["P1"]["completed_epoch_updates"] or
                 main["P3"]["run_total_window_exposures"] !=
                 main["P1"]["completed_epoch_window_exposures"])):
            main["P3"]["status"] = "invalid_unmatched_exposure"
    complete_main = {name: value for name, value in main.items()
        if value.get("status") == "completed"
        and (name == "P3" or int(value.get("epochs_completed", 0)) >= 5)}
    parent = max(complete_main, key=lambda name: (float(complete_main[name]["best_score"]),
                 -(("P1", "P2", "P3").index(name))), default="")
    if parent:
        parent_kind = "ast" if parent == "P2" else "sonics"
        parent_rate = chosen[parent_kind]
        parent_weights = Path(str(complete_main[parent]["weights"]))
        p4_steps = math.ceil((math.ceil(len(train) / 2) + len(train) // 2) / 4) * 5
        temporal_end = stages["temporal"]
        continued = execute(TrialSpec(f"{parent}_continued", parent_kind, parent_rate,
                          3 * parent_rate if parent_kind == "sonics" else 1e-4, 5,
                          max_updates=p4_steps, p4=True, p4_control=True, patience=0),
                          (time.time() + temporal_end) / 2, parent_weights)
        main["P4"] = execute(TrialSpec("P4", parent_kind, parent_rate,
                             3 * parent_rate if parent_kind == "sonics" else 1e-4,
                             5, max_updates=p4_steps, p4=True, patience=0), temporal_end, parent_weights)
        atomic_json(output / "p4_comparison.json", {"parent": parent,
            "parent_checkpoint": str(parent_weights), "optimizer_updates": p4_steps,
            "treatment": "partial_fake_bags_with_known_window_auxiliary",
            "control": "same_P4_window_schedule_with_matched_real_splices_mean_parent_loss_no_aux",
            "treatment_run_updates": main["P4"].get("run_total_updates", 0),
            "control_run_updates": continued.get("run_total_updates", 0),
            "treatment_window_exposures": main["P4"].get("run_total_window_exposures", 0),
            "control_window_exposures": continued.get("run_total_window_exposures", 0)})
        if (main["P4"].get("status") == "completed" and
                (main["P4"]["run_total_updates"] != continued["run_total_updates"] or
                 main["P4"]["run_total_window_exposures"] != continued["run_total_window_exposures"])):
            main["P4"]["status"] = "invalid_unmatched_exposure"
    candidates: dict[str, list[dict[str, str | float]]] = {}
    runtime: dict[str, float] = {}
    projections: dict[str, float] = {}
    for name in ("P1", "P2", "P3", "P4"):
        result = main.get(name)
        minimum_epochs = 5 if name in ("P1", "P2", "P4") else 1
        if (not result or result.get("status") != "completed"
                or int(result.get("epochs_completed", 0)) < minimum_epochs):
            continue
        prediction_path = output / "training" / name / f"epoch_{int(result['best_epoch']):03d}_dev_predictions.csv"
        predictions = read_predictions(prediction_path)
        write_predictions(output / f"{name}_dev_predictions.csv", predictions)
        candidates[name] = predictions
        runtime[name] = float(result["seconds"])
        updates = max(1, int(result["updates"]))
        projections[name] = runtime[name] / updates * math.ceil(updates * len(music_supervision_rows([*train_rows, *dev_rows])) / len(train))
        if name == "P4":
            parent_result = main[parent]
            parent_updates = max(1, int(parent_result["updates"]))
            projections[name] += (float(parent_result["seconds"]) / parent_updates
                                  * math.ceil(parent_updates * len(music_supervision_rows([*train_rows, *dev_rows])) / len(train)))
    profile = {name: {"search_seconds": runtime[name], "updates": main[name]["updates"],
                       "projected_complete_refit_seconds": projections[name]} for name in candidates}
    atomic_json(output / "throughput_forecast.json", profile)
    reserved = max(0.0, stages["refit"] - max(time.time(), stages["temporal"]))
    selection = select_development(dev_rows, g0, candidates, runtime, projections, reserved)
    selection_path = output / "selection.json"
    atomic_json(selection_path, selection)
    statuses = {"G0": {"status": "completed", "files": len(g0)}}
    statuses.update({name: {"status": main.get(name, {}).get("status", "not_run"),
                            "reason": "no_equal_completed_evidence" if name not in main else ""}
                     for name in ("P1", "P2", "P3", "P4")})
    atomic_json(output / "pipeline_status.json", statuses)
    trials_path = output / "trials.jsonl"
    temporary = trials_path.with_suffix(".tmp")
    with temporary.open("w", encoding="utf-8") as stream:
        for outcome in outcomes.values():
            stream.write(json.dumps(outcome, sort_keys=True, allow_nan=False) + "\n")
        for name, status in statuses.items():
            if name == "G0" or name not in main:
                stream.write(json.dumps({"name": name, **status}, sort_keys=True) + "\n")
    temporary.replace(trials_path)
    _write_epoch_artifacts(output, outcomes)
    selected = str(selection["selected"])
    if selected == "G0":
        budget_event(budget_path, "retained_g0", reason="no_candidate_passed")
        return {"final": "G0", "selection": selection, "sealed_opened": False}
    return refit_and_seal(args, stages, selected, parent, main, chosen, selection_path,
                          train_rows, dev_rows, g0, assets, device, output, budget_path)


def main() -> None:
    parser = argparse.ArgumentParser()
    for name in ("data", "baseline", "sonics", "ast", "output"):
        parser.add_argument(f"--{name}", type=Path, required=True)
    parser.add_argument("--deadline", type=float, required=True)
    args = parser.parse_args()
    try:
        result = run(args)
        atomic_json(args.output / "finished.json", {**result, "completed_unix": time.time(),
                    "actual_gpu_metrics": "measured_in_this_run" if torch.cuda.is_available() else "pending"})
    except Exception:  # noqa: BROAD_EXCEPT_OK
        args.output.mkdir(parents=True, exist_ok=True)
        (args.output / "failure.txt").write_text(__import__("traceback").format_exc(), encoding="utf-8")
        raise


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/dacon_trainable_v4/v4_training.py
"""Resumable optimizer primitives shared by the V4 training trials."""

from __future__ import annotations

import random
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Final, Literal

import numpy as np
import torch

CHECKPOINT_SECONDS: Final = 60.0
RunStatus = Literal["complete", "interrupted", "stopped_deadline"]
NamedParameters = Sequence[tuple[str, torch.nn.Parameter]]


@dataclass(frozen=True, slots=True)
class Batch:
    ids: tuple[str, ...]
    inputs: torch.Tensor
    targets: torch.Tensor
    sample_weights: torch.Tensor | None = None
    file_indices: torch.Tensor | None = None
    window_targets: torch.Tensor | None = None
    starts: tuple[tuple[int, ...], ...] = ()
    exposure_records: tuple[str, ...] = ()


@dataclass(frozen=True, slots=True)
class RunState:
    epoch: int = 0
    batch_index: int = 0
    optimizer_steps: int = 0
    exposures: dict[str, int] = field(default_factory=dict)
    window_exposures: int = 0
    exposure_records: tuple[str, ...] = ()


@dataclass(frozen=True, slots=True)
class RunResult:
    status: RunStatus
    state: RunState


def build_optimizer(
    encoder: NamedParameters,
    head: NamedParameters,
    *,
    encoder_lr: float,
    head_lr: float,
) -> torch.optim.AdamW:
    """Apply declared LR and bias/norm decay without omitting a trainable layer."""
    named = (*encoder, *head)
    if not named or len({id(parameter) for _, parameter in named}) != len(named):
        raise ValueError("optimizer parameters must be nonempty and unique")
    if not all(parameter.requires_grad for _, parameter in named):
        raise ValueError("full fine-tuning requires every learned parameter")
    groups: list[dict[str, float | list[torch.nn.Parameter]]] = []
    for parameters, learning_rate in ((encoder, encoder_lr), (head, head_lr)):
        for decay in (0.01, 0.0):
            selected = [
                parameter
                for _, parameter in parameters
                if (parameter.ndim >= 2) == (decay > 0)
            ]
            if selected:
                groups.append({"params": selected, "lr": learning_rate, "weight_decay": decay})
    return torch.optim.AdamW(groups, betas=(0.9, 0.999), eps=1e-8)


def build_scheduler(
    optimizer: torch.optim.Optimizer,
    *,
    steps_per_epoch: int,
    planned_epochs: int = 30,
) -> torch.optim.lr_scheduler.LambdaLR:
    """Warm one full epoch, then decay over the fixed 30-epoch horizon."""
    if steps_per_epoch < 1 or planned_epochs < 2:
        raise ValueError("scheduler requires positive steps and at least two epochs")
    total_steps = steps_per_epoch * planned_epochs

    def multiplier(step: int) -> float:
        if step < steps_per_epoch:
            return 0.1 + 0.9 * step / steps_per_epoch
        progress = (step - steps_per_epoch) / (total_steps - steps_per_epoch)
        return 0.1 + 0.45 * (1.0 + np.cos(np.pi * min(progress, 1.0)))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, multiplier)


def _rng_state() -> dict[str, torch.Tensor | tuple | list[torch.Tensor] | None]:
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }


def _restore_rng(state: Mapping[str, torch.Tensor | tuple | list[torch.Tensor] | None]) -> None:
    python_state = state["python"]
    numpy_state = state["numpy"]
    torch_state = state["torch"]
    cuda_state = state["cuda"]
    assert isinstance(python_state, tuple)
    assert isinstance(numpy_state, tuple)
    assert isinstance(torch_state, torch.Tensor)
    random.setstate(python_state)
    np.random.set_state(numpy_state)
    torch.set_rng_state(torch_state)
    if cuda_state is not None and torch.cuda.is_available():
        assert isinstance(cuda_state, list)
        torch.cuda.set_rng_state_all(cuda_state)


def save_checkpoint(
    path: Path,
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.LRScheduler,
    state: RunState,
    contract: str,
) -> None:
    """Atomically bind weights, optimizer, scheduler, RNG, and exposure progress."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(
        {
            "contract": contract,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "state": asdict(state),
            "rng": _rng_state(),
        },
        temporary,
    )
    temporary.replace(path)


def load_checkpoint(
    path: Path,
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.LRScheduler,
    contract: str,
) -> RunState:
    payload = torch.load(path, map_location="cpu", weights_only=False)
    if payload["contract"] != contract:
        raise ValueError("checkpoint contract mismatch")
    model.load_state_dict(payload["model"], strict=True)
    optimizer.load_state_dict(payload["optimizer"])
    scheduler.load_state_dict(payload["scheduler"])
    _restore_rng(payload["rng"])
    raw = payload["state"]
    return RunState(
        epoch=int(raw["epoch"]),
        batch_index=int(raw["batch_index"]),
        optimizer_steps=int(raw["optimizer_steps"]),
        exposures={str(key): int(value) for key, value in raw["exposures"].items()},
        window_exposures=int(raw.get("window_exposures", 0)),
        exposure_records=tuple(map(str, raw.get("exposure_records", ()))),
    )


def _default_loss(model: torch.nn.Module, batch: Batch) -> torch.Tensor:
    logits = model(batch.inputs).reshape(-1)
    losses = torch.nn.functional.binary_cross_entropy_with_logits(
        logits, batch.targets, reduction="none"
    )
    return losses.mean() if batch.sample_weights is None else (
        losses * batch.sample_weights
    ).sum() / batch.sample_weights.sum()


def run_batches(
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.LRScheduler,
    batches: Sequence[Batch],
    state: RunState,
    checkpoint: Path,
    deadline: float,
    *,
    now: Callable[[], float] = time.time,
    max_updates: int | None = None,
    contract: str = "test-contract",
    loss_fn: Callable[[torch.nn.Module, Batch], torch.Tensor] = _default_loss,
    accumulation: int = 1,
) -> RunResult:
    """Run deterministic batches and durably stop at a budget or requested boundary."""
    model.train()
    exposures = dict(state.exposures)
    current = state
    saved_at = now()
    if accumulation < 1:
        raise ValueError("accumulation must be positive")
    for start in range(state.batch_index, len(batches), accumulation):
        if now() >= deadline:
            save_checkpoint(checkpoint, model, optimizer, scheduler, current, contract)
            return RunResult("stopped_deadline", current)
        optimizer.zero_grad(set_to_none=True)
        try:
            group = [batches[index] for index in range(start, min(start + accumulation, len(batches)))]
        except TimeoutError:
            save_checkpoint(checkpoint, model, optimizer, scheduler, current, contract)
            return RunResult("stopped_deadline", current)
        if now() >= deadline:
            save_checkpoint(checkpoint, model, optimizer, scheduler, current, contract)
            return RunResult("stopped_deadline", current)
        for batch in group:
            loss = loss_fn(model, batch) / len(group)
            if not torch.isfinite(loss):
                raise FloatingPointError("non-finite training loss")
            loss.backward()
        gradient_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        if not torch.isfinite(gradient_norm):
            raise FloatingPointError("non-finite training gradient")
        optimizer.step()
        scheduler.step()
        for batch in group:
            for identifier in batch.ids:
                exposures[identifier] = exposures.get(identifier, 0) + 1
        current = RunState(
            epoch=state.epoch,
            batch_index=start + len(group),
            optimizer_steps=current.optimizer_steps + 1,
            exposures=exposures.copy(),
            window_exposures=current.window_exposures + sum(batch.inputs.shape[0] for batch in group),
            exposure_records=current.exposure_records + tuple(
                record for batch in group for record in batch.exposure_records),
        )
        if now() - saved_at >= CHECKPOINT_SECONDS:
            save_checkpoint(checkpoint, model, optimizer, scheduler, current, contract)
            saved_at = now()
        if (max_updates is not None and current.optimizer_steps - state.optimizer_steps >= max_updates
                and current.batch_index < len(batches)):
            save_checkpoint(checkpoint, model, optimizer, scheduler, current, contract)
            return RunResult("interrupted", current)
    save_checkpoint(checkpoint, model, optimizer, scheduler, current, contract)
    return RunResult("complete", current)


In [ ]:
%%writefile /content/dacon_trainable_v4/v4_training_refit.py
"""Recipe lock, complete eligible-data refit, and one-way sealed evaluation."""

from __future__ import annotations

import csv
import json
import time
from collections.abc import Mapping
from dataclasses import replace
from pathlib import Path

import torch

from v4_batches import music_supervision_rows
from v4_data import load_manifest
from v4_model import load_model, parameter_groups
from v4_selection import (accept_sealed, mark_sealed_opened, write_recipe_lock,
    write_sealed_inference_receipt, write_weight_lock)
from v4_training_io import atomic_json, budget_event, digest, g0_predictions
from v4_training_trials import TrialSpec, predict_rows, train_trial, write_predictions


def _hashes(paths: list[Path]) -> dict[str, str]:
    return {str(path): digest(path) for path in paths if path.is_file()}


def _last_exposures(result: Mapping) -> dict[str, int]:
    reports = result.get("reports", [])
    return {} if not reports else {str(key): int(value) for key, value in reports[-1]["cumulative_exposures"].items()}


def refit_and_seal(args, stage_deadlines: Mapping[str, float], selected: str, parent: str,
                   main: Mapping[str, Mapping], chosen: Mapping[str, float], selection_path: Path,
                   train_rows: list[dict[str, str]], dev_rows: list[dict[str, str]],
                   g0: list[dict[str, str | float]], assets: Mapping[str, Path],
                   device: torch.device, output: Path, budget_path: Path) -> dict:
    provenance = json.loads((args.data / "provenance.json").read_text(encoding="utf-8"))
    provenance_by_id = {row["ID"]: row for row in provenance["sources"]}
    resolved = []
    for row in [*train_rows, *dev_rows]:
        full_path = str(provenance_by_id[row["ID"]].get("full_path", ""))
        resolved.append({**row, "path": full_path or row["path"],
                         "refit_path_source": "full_original" if full_path else "declared_derived"})
    eligible = music_supervision_rows(resolved)
    identifiers = [row["ID"] for row in eligible]
    if len(set(identifiers)) != len(identifiers):
        raise ValueError("eligible train+dev refit IDs must be unique")
    eligible_manifest = output / "eligible_refit_manifest.csv"
    with eligible_manifest.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=("ID", "path", "path_source", "sha256"))
        writer.writeheader()
        writer.writerows({"ID": row["ID"], "path": row["path"],
                          "path_source": row["refit_path_source"],
                          "sha256": digest(args.data / row["path"])} for row in eligible)
    kind = "ast" if selected == "P2" or (selected == "P4" and parent == "P2") else "sonics"
    rate = chosen[kind]
    stage_epochs = ({parent: int(main[parent]["best_epoch"]),
                     "P4": int(main["P4"]["best_epoch"])} if selected == "P4"
                    else {selected: int(main[selected]["best_epoch"])})
    epoch_count = sum(stage_epochs.values())
    data_files = [args.data / name for name in ("train.csv", "dev.csv", "protocol.json",
                                                 "provenance.json", "integrity_report.json")]
    code_files = [Path(name) for name in ("train_v4.py", "v4_model.py",
        "v4_model_components.py", "v4_training.py", "v4_training_data.py",
        "v4_training_trials.py", "v4_training_refit.py", "v4_training_io.py", "v4_batches.py",
        "v4_data.py", "v4_evaluation.py", "v4_selection.py", "graph_features.py",
        "graph_fusion.py", "graph_scoring.py", "v2_scoring.py", "recovery_sonics.py",
        "ast_multitask.py", "v2_data.py")]
    recipe = {"selected": selected, "parent": parent or selected,
              "treatment": "parent_then_temporal_bag" if selected == "P4" else "full_fine_tune",
              "learning_rate": {"encoder": rate, "head": 3 * rate if kind == "sonics" else 1e-4},
              "scheduler": {"warmup_epochs": 1, "horizon_epochs": 30, "minimum_multiplier": .1},
              "epoch_count": epoch_count, "stage_epochs": stage_epochs,
              "data_hashes": _hashes(data_files), "source_hashes": _hashes(code_files),
              "pooling": "normalized_logsumexp_temperature_2" if selected == "P4" else "mean_probability",
              "seeds": {"training": 20260913, "bootstrap": 20260913},
              "gate_thresholds": {"D": .005, "eer_regression": .01, "auc_regression": .01,
                  "bootstrap_win": .80, "bootstrap_p10": 0, "bootstrap_valid": .90,
                  "controlled_file_eer": .02, "refit_safety_factor": 1.25}}
    recipe_path = output / "recipe_lock.json"
    write_recipe_lock(recipe_path, selection_path, eligible_manifest, recipe)
    refit_started = time.time()
    refit_results: list[dict] = []

    def fit(spec: TrialSpec, state: Path | None = None) -> dict:
        model_asset = assets[spec.kind] / ("pytorch_model.bin" if spec.kind == "sonics" else "model.safetensors")
        contract_files = [args.data / "train.csv", args.data / "dev.csv", eligible_manifest,
                          model_asset, *code_files]
        spec = replace(spec, contract_hash="|".join(digest(path) for path in contract_files))
        loaded = load_model(spec.kind, assets[spec.kind], device, state_path=state)
        groups = parameter_groups(loaded)
        result = train_trial(loaded, (groups.encoder, groups.head), spec, eligible, dev_rows, g0,
                             args.data, output / "refit" / spec.name,
                             min(args.deadline, stage_deadlines["refit"]))
        refit_results.append(result)
        return result

    if selected == "P4":
        parent_spec = TrialSpec("parent", kind, rate, 3 * rate if kind == "sonics" else 1e-4,
                                stage_epochs[parent], p3=parent == "P3", patience=0)
        first = fit(parent_spec)
        if first.get("status") == "completed":
            final = fit(TrialSpec("P4", kind, rate, 3 * rate if kind == "sonics" else 1e-4,
                                  stage_epochs["P4"], p4=True, patience=0),
                        Path(str(first["final_weights"])))
        else:
            final = first
    else:
        final = fit(TrialSpec(selected, kind, rate, 3 * rate if kind == "sonics" else 1e-4,
                              epoch_count, p3=selected == "P3", patience=0))
    complete = (len(refit_results) == len(stage_epochs) and all(
        result.get("status") == "completed" and int(result.get("epochs_completed", 0)) == epochs
        for result, epochs in zip(refit_results, stage_epochs.values(), strict=True)))
    actual: dict[str, int] = {identifier: 0 for identifier in identifiers}
    for result in refit_results:
        for identifier, count in _last_exposures(result).items():
            if identifier in actual:
                actual[identifier] += count
    exposures_complete = complete and all(actual[identifier] == epoch_count for identifier in identifiers)
    refit_manifest = output / "refit_manifest.csv"
    with refit_manifest.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=("ID", "expected_exposures", "actual_exposures"))
        writer.writeheader()
        writer.writerows({"ID": identifier, "expected_exposures": epoch_count,
                          "actual_exposures": actual[identifier]} for identifier in identifiers)
    audit = {"completed": complete, "exposures_complete": exposures_complete,
             "prescribed_epochs": epoch_count,
             "completed_epochs": sum(int(result.get("epochs_completed", 0)) for result in refit_results),
             "started_unix": refit_started, "completed_unix": time.time(), "stages": refit_results}
    atomic_json(output / "final_weight_audit.json", audit)
    if not complete or not exposures_complete:
        budget_event(budget_path, "refit_incomplete")
        return {"final": "G0", "reason": "refit_incomplete", "sealed_opened": False}
    weights = Path(str(final["final_weights"]))
    weight_lock = output / "weight_lock.json"
    write_weight_lock(weight_lock, recipe_path, weights, code_files, refit_manifest, audit)
    opened = output / "sealed_opened.json"
    mark_sealed_opened(opened, weight_lock, weights)
    sealed_started = time.time()
    sealed_rows = load_manifest(args.data, "sealed")
    sealed_g0 = g0_predictions(sealed_rows, args.data, args.baseline, output / "sealed",
                               min(args.deadline, stage_deadlines["sealed"]))
    loaded = load_model(kind, assets[kind], device, state_path=weights)
    candidate = predict_rows(loaded, sealed_rows, args.data, sealed_g0, p4=selected == "P4",
                             deadline=min(args.deadline, stage_deadlines["sealed"]))
    candidate_path = output / f"{selected}_sealed_predictions.csv"
    write_predictions(candidate_path, candidate)
    g0_path = output / "sealed" / "G0_predictions.csv"
    receipt = output / "sealed_inference_receipt.json"
    write_sealed_inference_receipt(receipt, weight_lock, weights, opened, g0_path, candidate_path,
                                   sealed_started, time.time())
    acceptance = accept_sealed(sealed_rows, selected, weight_lock, weights, opened, receipt)
    atomic_json(output / "sealed_acceptance.json", acceptance)
    budget_event(budget_path, "sealed_complete", accepted=bool(acceptance["accepted"]))
    return {"final": acceptance["final"], "selection": acceptance, "sealed_opened": True}


In [ ]:
%%writefile /content/dacon_trainable_v4/v4_training_trials.py
"""One bounded full-fine-tuning trial and full-file prediction."""

from __future__ import annotations

import csv
import hashlib
import json
import math
import time
from collections.abc import Mapping, Sequence
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Literal

import torch
from safetensors.torch import save_file

from v4_batches import music_supervision_rows
from v4_data import load_audio
from v4_evaluation import evaluate_development
from v4_model import LoadedModel, PreparedWindows, pool_bag_logits, predict_files, window_logits
from v4_training import Batch, RunState, build_optimizer, build_scheduler, load_checkpoint, run_batches, save_checkpoint
from v4_training_data import EpochBatches

Row = Mapping[str, str]


@dataclass(frozen=True, slots=True)
class TrialSpec:
    name: str
    kind: Literal["sonics", "ast"]
    encoder_lr: float
    head_lr: float
    epochs: int
    max_updates: int | None = None
    p3: bool = False
    p4: bool = False
    p4_control: bool = False
    patience: int = 5
    contract_hash: str = ""


def _hash(value: Mapping[str, str | int | float | bool | None]) -> str:
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def _file_digest(path: Path) -> str:
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def _save_weights(path: Path, model: torch.nn.Module) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    save_file({name: value.detach().cpu() for name, value in model.state_dict().items()}, temporary)
    temporary.replace(path)


def _atomic_payload(path: Path, value: Mapping) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, sort_keys=True, allow_nan=False), encoding="utf-8")
    temporary.replace(path)


def _finish(output: Path, contract: str, result: dict) -> dict:
    artifacts = {}
    for key in ("weights", "final_weights"):
        path = Path(str(result[key])) if result.get(key) else None
        if path is not None and path.is_file():
            artifacts[key] = _file_digest(path)
    _atomic_payload(output / "terminal_result.json",
                    {"contract": contract, "artifacts": artifacts, "result": result})
    return result


def _load_terminal(output: Path, contract: str) -> dict | None:
    path = output / "terminal_result.json"
    if not path.is_file():
        return None
    payload = json.loads(path.read_text(encoding="utf-8"))
    if payload.get("contract") != contract:
        raise ValueError("terminal result contract mismatch")
    result = payload["result"]
    for key, expected in payload.get("artifacts", {}).items():
        artifact = Path(str(result[key]))
        if not artifact.is_file() or _file_digest(artifact) != expected:
            raise ValueError(f"terminal result {key} hash mismatch")
    return result


def _write_exposure_records(output: Path, state: RunState) -> Path:
    path = output / "exposure_records.json"
    _atomic_payload(path, {"records": [json.loads(value) for value in state.exposure_records]})
    return path


def _weighted_music_loss(logits: torch.Tensor, metadata: PreparedWindows, targets: torch.Tensor,
                         weights: torch.Tensor, *, p4: bool, p4_control: bool,
                         file_count: int) -> torch.Tensor:
    file_indices = metadata.file_indices.to(logits.device)
    if not p4 or p4_control:
        losses = torch.nn.functional.binary_cross_entropy_with_logits(
            logits, targets[file_indices], reduction="none")
        if p4_control:
            per_bag = [losses[file_indices == index].mean() * weights[index]
                       for index in range(file_count)]
            return torch.stack(per_bag).sum()
        return (losses * weights[file_indices]).sum()
    pooled = pool_bag_logits(logits, file_indices, file_count)
    result = (torch.nn.functional.binary_cross_entropy_with_logits(
        pooled, targets, reduction="none") * weights).sum()
    window_targets = metadata.window_targets.to(logits.device)
    known = torch.isfinite(window_targets)
    if torch.any(known):
        auxiliary = torch.nn.functional.binary_cross_entropy_with_logits(
            logits[known], window_targets[known], reduction="none")
        known_files = file_indices[known]
        per_bag = []
        for index in range(file_count):
            selected = auxiliary[known_files == index]
            if selected.numel():
                per_bag.append(weights[index] * selected.mean())
        if per_bag:
            result = result + 0.5 * torch.stack(per_bag).sum()
    return result


def _early_stop(epoch: int, patience_epoch: int, patience: int) -> bool:
    return patience > 0 and epoch >= 5 and epoch - patience_epoch > patience


def predict_rows(
    loaded: LoadedModel,
    rows: Sequence[Row],
    root: Path,
    baseline: Sequence[Mapping[str, str | float]],
    *,
    p4: bool,
    deadline: float,
) -> list[dict[str, str | float]]:
    baseline_by_id = {str(row["ID"]): row for row in baseline}
    if len(baseline_by_id) != len(rows):
        raise ValueError("baseline prediction coverage mismatch")
    result: list[dict[str, str | float]] = []
    loaded.model.eval()
    for offset in range(0, len(rows), 8):
        if time.time() >= deadline:
            raise TimeoutError("deadline reached during full-file prediction")
        selected = rows[offset : offset + 8]
        values = predict_files(loaded, [load_audio(root / row["path"]) for row in selected], p4=p4).probabilities
        for row, probability in zip(selected, values.tolist(), strict=True):
            base = baseline_by_id[str(row["ID"])]
            result.append({"ID": row["ID"],
                "FILE_FAKE_PROB": max(float(base["VOICE_PRESENT_PROB"]) * float(base["VOICE_FAKE_PROB"]),
                                      float(base["MUSIC_PRESENT_PROB"]) * probability),
                "VOICE_FAKE_PROB": base["VOICE_FAKE_PROB"], "MUSIC_FAKE_PROB": probability,
                "VOICE_PRESENT_PROB": base["VOICE_PRESENT_PROB"],
                "MUSIC_PRESENT_PROB": base["MUSIC_PRESENT_PROB"]})
    return result


def write_predictions(path: Path, rows: Sequence[Mapping[str, str | float]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=("ID", "FILE_FAKE_PROB", "VOICE_FAKE_PROB",
            "MUSIC_FAKE_PROB", "VOICE_PRESENT_PROB", "MUSIC_PRESENT_PROB"))
        writer.writeheader()
        writer.writerows(rows)


def profile_update(
    loaded: LoadedModel,
    groups: tuple[Sequence[tuple[str, torch.nn.Parameter]], Sequence[tuple[str, torch.nn.Parameter]]],
    rows: Sequence[Row],
    data_root: Path,
    inference_row: Row,
    codec_cache: Path,
    deadline: float,
) -> dict[str, str | int | float]:
    """Measure real preparation, one full-parameter update, and one full-file inference."""
    eligible = music_supervision_rows(rows)
    microbatch = 2
    batches = EpochBatches(loaded, eligible, data_root, 0, microbatch=microbatch, p3=False,
                           p4=False, p4_control=False, codec_cache=data_root / ".unused", deadline=deadline)
    prepared_at = time.perf_counter()
    batch = batches[0]
    preparation_seconds = time.perf_counter() - prepared_at
    p3_preparation_seconds = 0.0
    if loaded.kind == "sonics":
        p3_batches = EpochBatches(loaded, eligible, data_root, 0, microbatch=microbatch, p3=True,
                                  p4=False, p4_control=False, codec_cache=codec_cache,
                                  deadline=deadline)
        p3_at = time.perf_counter()
        p3_batches[0]
        p3_preparation_seconds = time.perf_counter() - p3_at
    before = {name: value.detach().cpu().clone() for name, value in loaded.model.named_parameters()}
    optimizer = build_optimizer(groups[0], groups[1], encoder_lr=1e-5 if loaded.kind == "sonics" else 1e-6,
                                head_lr=3e-5 if loaded.kind == "sonics" else 1e-4)
    optimizer.zero_grad(set_to_none=True)
    if next(loaded.model.parameters()).is_cuda:
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    update_at = time.perf_counter()
    logits = loaded.model(batch.inputs.to(next(loaded.model.parameters()).device)).reshape(-1).float()
    loss = torch.nn.functional.binary_cross_entropy_with_logits(
        logits, batch.targets.to(logits.device))
    loss.backward()
    gradient_norm = float(torch.nn.utils.clip_grad_norm_(loaded.model.parameters(), 1.0))
    optimizer.step()
    if next(loaded.model.parameters()).is_cuda:
        torch.cuda.synchronize()
    update_seconds = time.perf_counter() - update_at
    absolute_update = sum(float((parameter.detach().cpu() - before[name]).abs().sum())
                          for name, parameter in loaded.model.named_parameters())
    inference_at = time.perf_counter()
    predict_files(loaded, [load_audio(data_root / inference_row["path"])], p4=False)
    if next(loaded.model.parameters()).is_cuda:
        torch.cuda.synchronize()
    inference_seconds = time.perf_counter() - inference_at
    return {"kind": loaded.kind, "sample_file": inference_row["ID"],
            "preparation_seconds": preparation_seconds, "update_seconds": update_seconds,
            "p3_preparation_seconds": p3_preparation_seconds,
            "full_file_inference_seconds": inference_seconds, "gradient_norm": gradient_norm,
            "absolute_parameter_update": absolute_update,
            "trainable_parameters": sum(parameter.numel() for parameter in loaded.model.parameters()),
            "peak_allocated_gib": torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else 0.0}


def train_trial(
    loaded: LoadedModel,
    groups: tuple[Sequence[tuple[str, torch.nn.Parameter]], Sequence[tuple[str, torch.nn.Parameter]]],
    spec: TrialSpec,
    train_rows: Sequence[Row],
    dev_rows: Sequence[Row],
    baseline: Sequence[Mapping[str, str | float]],
    data_root: Path,
    output: Path,
    deadline: float,
) -> dict[str, str | int | float | bool | list | dict]:
    eligible = music_supervision_rows(train_rows)
    if not eligible:
        raise ValueError("training manifest has no eligible music supervision")
    microbatch, accumulation = ((2, 4) if spec.p4 else (8, 4))
    probe_batches = EpochBatches(loaded, eligible, data_root, 0, microbatch=microbatch,
        p3=spec.p3, p4=spec.p4, p4_control=spec.p4_control,
        codec_cache=output / "codec_cache", deadline=deadline)
    steps_per_epoch = math.ceil(len(probe_batches) / accumulation)
    optimizer = build_optimizer(groups[0], groups[1], encoder_lr=spec.encoder_lr, head_lr=spec.head_lr)
    scheduler = build_scheduler(optimizer, steps_per_epoch=steps_per_epoch)
    contract = _hash({**asdict(spec), "train_ids": hashlib.sha256("\n".join(row["ID"] for row in eligible).encode()).hexdigest()})
    terminal = _load_terminal(output, contract)
    if terminal is not None:
        return terminal
    latest, best = output / "latest.pt", output / "best.safetensors"
    state = load_checkpoint(latest, loaded.model, optimizer, scheduler, contract) if latest.is_file() else RunState()
    reports_path = output / "epochs.json"
    reports = json.loads(reports_path.read_text()) if reports_path.is_file() else []
    best_metadata = output / "best.json"
    if best_metadata.is_file():
        saved_best = json.loads(best_metadata.read_text(encoding="utf-8"))
        if saved_best["contract"] != contract or not best.is_file():
            raise ValueError("best checkpoint contract mismatch")
        if _file_digest(best) != saved_best["checkpoint_sha256"]:
            raise ValueError("best checkpoint hash mismatch")
        best_score, best_epoch = float(saved_best["score"]), int(saved_best["epoch"])
        best_updates = int(saved_best["optimizer_steps"])
        best_windows = int(saved_best["window_exposures"])
    else:
        best_score, best_epoch, best_updates, best_windows = float("-inf"), 0, 0, 0
    patience_score, patience_epoch = float("-inf"), 0
    for report in reports:
        score = float(report["rank_score"])
        if score >= patience_score + 0.002:
            patience_score, patience_epoch = score, int(report["epoch"])
    started = time.perf_counter()

    def complete_fixed_update() -> dict:
        predictions = predict_rows(loaded, dev_rows, data_root, baseline,
                                   p4=spec.p4 and not spec.p4_control,
                                   deadline=deadline)
        write_predictions(output / "fixed_update_dev_predictions.csv", predictions)
        evaluation = evaluate_development(dev_rows, baseline, predictions)
        final = output / "final.safetensors"
        _save_weights(final, loaded.model)
        value = {"name": spec.name, "status": "completed", "fixed_updates": True,
            "updates": state.optimizer_steps, "epochs_completed": len(reports),
            "window_exposures": state.window_exposures, "run_total_updates": state.optimizer_steps,
            "run_total_window_exposures": state.window_exposures, "best_epoch": 0,
            "best_score": float(evaluation["rank_score"]), "weights": str(final),
            "reports": reports, "evaluation": evaluation,
            "exposure_records_path": str(_write_exposure_records(output, state)),
            "seconds": time.perf_counter() - started,
            "trainable_parameters": sum(p.numel() for p in loaded.model.parameters())}
        return _finish(output, contract, value)

    while state.epoch < spec.epochs:
        batches = EpochBatches(loaded, eligible, data_root, state.epoch, microbatch=microbatch,
            p3=spec.p3, p4=spec.p4, p4_control=spec.p4_control,
            codec_cache=output / "codec_cache", deadline=deadline)

        def loss_fn(_model: torch.nn.Module, batch: Batch) -> torch.Tensor:
            if batch.file_indices is None or batch.window_targets is None:
                raise RuntimeError("prepared batch metadata is missing")
            metadata = PreparedWindows(batch.inputs, batch.file_indices, batch.starts, batch.window_targets)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=next(loaded.model.parameters()).is_cuda):
                logits = window_logits(loaded, metadata)
            logits = logits.float()
            targets = batch.targets.to(logits.device)
            weights = batch.sample_weights.to(logits.device)
            return _weighted_music_loss(logits, metadata, targets, weights, p4=spec.p4,
                                        p4_control=spec.p4_control, file_count=len(batch.ids))

        remaining = None if spec.max_updates is None else spec.max_updates - state.optimizer_steps
        if remaining is not None and remaining <= 0:
            return complete_fixed_update()
        result = run_batches(loaded.model, optimizer, scheduler, batches, state, latest, deadline,
                             max_updates=remaining, contract=contract, loss_fn=loss_fn, accumulation=accumulation)
        state = result.state
        exposure_path = _write_exposure_records(output, state)
        fixed_complete = (result.status == "interrupted" and spec.max_updates is not None
                          and state.optimizer_steps >= spec.max_updates)
        if result.status != "complete" and not fixed_complete:
            if result.status == "stopped_deadline" and len(reports) >= 5 and best.is_file():
                return _finish(output, contract, {"name": spec.name, "status": "completed", "resource_censored": True,
                        "updates": best_updates, "epochs_completed": len(reports),
                        "completed_epoch_updates": int(reports[-1]["optimizer_steps"]),
                        "completed_epoch_window_exposures": int(reports[-1]["window_exposures"]),
                        "run_total_updates": state.optimizer_steps,
                        "run_total_window_exposures": state.window_exposures,
                        "window_exposures": best_windows, "best_epoch": best_epoch,
                        "best_score": best_score, "weights": str(best), "reports": reports,
                        "exposure_records_path": str(exposure_path),
                        "seconds": time.perf_counter() - started,
                        "trainable_parameters": sum(p.numel() for p in loaded.model.parameters())})
            return {"name": spec.name, "status": result.status, "updates": state.optimizer_steps,
                    "epochs_completed": len(reports), "window_exposures": state.window_exposures,
                    "exposure_records_path": str(exposure_path),
                    "seconds": time.perf_counter() - started}
        if fixed_complete:
            return complete_fixed_update()
        expected = {row["ID"] for row in eligible}
        previous = {key: int(value) for key, value in (reports[-1]["cumulative_exposures"].items() if reports else ())}
        if any(state.exposures.get(identifier, 0) - previous.get(identifier, 0) != 1 for identifier in expected):
            raise RuntimeError("complete epoch did not expose every eligible file exactly once")
        predictions = predict_rows(loaded, dev_rows, data_root, baseline,
                                   p4=spec.p4 and not spec.p4_control, deadline=deadline)
        write_predictions(output / f"epoch_{state.epoch + 1:03d}_dev_predictions.csv", predictions)
        evaluation = evaluate_development(dev_rows, baseline, predictions)
        score = float(evaluation["rank_score"])
        report = {"epoch": state.epoch + 1, "optimizer_steps": state.optimizer_steps,
                  "window_exposures": state.window_exposures,
                  "rank_score": score, "evaluation": evaluation,
                  "cumulative_exposures": state.exposures}
        reports.append(report)
        reports_path.parent.mkdir(parents=True, exist_ok=True)
        reports_path.write_text(json.dumps(reports, allow_nan=False), encoding="utf-8")
        if score > best_score or not best.is_file():
            temporary = best.with_suffix(".tmp")
            save_file({name: value.detach().cpu() for name, value in loaded.model.state_dict().items()}, temporary)
            temporary.replace(best)
            best_score, best_epoch = score, state.epoch + 1
            best_updates, best_windows = state.optimizer_steps, state.window_exposures
            _atomic_payload(best_metadata, {"contract": contract, "epoch": best_epoch,
                "optimizer_steps": state.optimizer_steps, "score": best_score,
                "window_exposures": state.window_exposures,
                "checkpoint_sha256": _file_digest(best)})
        if score >= patience_score + 0.002:
            patience_score, patience_epoch = score, state.epoch + 1
        state = RunState(state.epoch + 1, 0, state.optimizer_steps, state.exposures,
                         state.window_exposures, state.exposure_records)
        save_checkpoint(latest, loaded.model, optimizer, scheduler, state, contract)
        if spec.max_updates is not None and state.optimizer_steps >= spec.max_updates:
            break
        if _early_stop(state.epoch, patience_epoch, spec.patience):
            break
    final = output / "final.safetensors"
    _save_weights(final, loaded.model)
    return _finish(output, contract, {"name": spec.name, "status": "completed", "updates": best_updates,
            "run_total_updates": state.optimizer_steps,
            "run_total_window_exposures": state.window_exposures,
            "completed_epoch_updates": int(reports[-1]["optimizer_steps"]),
            "completed_epoch_window_exposures": int(reports[-1]["window_exposures"]),
            "epochs_completed": len(reports), "best_epoch": best_epoch,
            "best_score": best_score, "weights": str(best), "final_weights": str(final), "reports": reports,
            "exposure_records_path": str(_write_exposure_records(output, state)),
            "seconds": time.perf_counter() - started, "window_exposures": best_windows,
            "trainable_parameters": sum(p.numel() for p in loaded.model.parameters())})


In [ ]:
%%writefile /content/dacon_trainable_v4/v4_training_data.py
"""Lazy CPU batches for full-source V4 epochs."""

from __future__ import annotations

import math
import json
import time
from collections import Counter
from collections.abc import Mapping, Sequence
from functools import lru_cache
from pathlib import Path

import numpy as np
import torch

from v4_batches import (apply_gain, epoch_crop, epoch_crop_start, epoch_order, p3_augment,
                        p3_variant, p4_splice, p4_starts)
from v4_data import load_audio
from v4_model import LoadedModel, prepare_windows
from v4_training import Batch

Row = Mapping[str, str]


@lru_cache(maxsize=None)
def _audio(path: Path) -> np.ndarray:
    """Cache only deterministic decoded PCM, never learned representations."""
    return load_audio(path)


def _weights(rows: Sequence[Row]) -> dict[str, float]:
    def provider(row: Row) -> str:
        parts = row["source"].split("/")
        return parts[1] if len(parts) >= 3 and parts[0].lower() == "echoes" else row["source"]

    real = [row for row in rows if row["MUSIC_FAKE"] == "0"]
    regular = [row for row in real if "nsynth" not in row["source"].lower()]
    nsynth = [row for row in real if "nsynth" in row["source"].lower()]
    fake = [row for row in rows if row["MUSIC_FAKE"] == "1"]
    fake_sources = Counter(provider(row) for row in fake)
    if not regular or not fake or not fake_sources:
        raise ValueError("weighted training requires real and fake music from named providers")
    weights = {row["ID"]: (0.45 if nsynth else 0.5) / len(regular) for row in regular}
    if nsynth:
        weights.update({row["ID"]: 0.05 / len(nsynth) for row in nsynth})
    provider_mass = 0.5 / len(fake_sources)
    weights.update({row["ID"]: provider_mass / fake_sources[provider(row)] for row in fake})
    return weights


class EpochBatches(Sequence[Batch]):
    """Generate one CPU microbatch on demand so an epoch is never retained in RAM/GPU tensors."""

    def __init__(
        self,
        loaded: LoadedModel,
        rows: Sequence[Row],
        root: Path,
        epoch: int,
        *,
        microbatch: int,
        p3: bool,
        p4: bool,
        p4_control: bool = False,
        codec_cache: Path,
        deadline: float,
    ) -> None:
        self.loaded = loaded
        self.rows = epoch_order(rows, epoch)
        self.root = root
        self.epoch = epoch
        self.microbatch = microbatch
        self.p3 = p3
        self.p4 = p4
        self.p4_control = p4_control
        self.codec_cache = codec_cache
        self.deadline = deadline
        self.weights = _weights(rows)
        self.original_batches = math.ceil(len(rows) / microbatch)
        self.splice_pairs = len(rows) // 2 if p4 else 0
        self.real = [row for row in self.rows if row["MUSIC_FAKE"] == "0"]
        self.fake = [row for row in self.rows if row["MUSIC_FAKE"] == "1"]

    def __len__(self) -> int:
        return self.original_batches + self.splice_pairs

    def __getitem__(self, index: int | slice) -> Batch | list[Batch]:
        if isinstance(index, slice):
            return [self[position] for position in range(*index.indices(len(self)))]
        if index < 0:
            index += len(self)
        if not 0 <= index < len(self):
            raise IndexError(index)
        if time.time() >= self.deadline:
            raise TimeoutError("deadline reached during CPU batch preparation")
        return self._original(index) if index < self.original_batches else self._splice(index - self.original_batches)

    def _original(self, index: int) -> Batch:
        selected = self.rows[index * self.microbatch : (index + 1) * self.microbatch]
        audios = [_audio(self.root / row["path"]) for row in selected]
        if self.p4:
            starts = [p4_starts(len(audio), self.loaded.window_samples, row["ID"], self.epoch)
                      for audio, row in zip(audios, selected, strict=True)]
            prepared = prepare_windows(self.loaded, audios, starts=starts)
        else:
            crops = [apply_gain(epoch_crop(audio, row["ID"], self.epoch, self.loaded.window_samples),
                                row["ID"], self.epoch)
                     for audio, row in zip(audios, selected, strict=True)]
            if self.p3:
                crops = [p3_augment(audio, row["ID"], self.epoch, str(self.codec_cache))
                         for audio, row in zip(crops, selected, strict=True)]
            prepared = prepare_windows(self.loaded, crops)
        scale = 0.5 if self.p4 else 1.0
        records = tuple(json.dumps({"ID": row["ID"], "epoch": self.epoch,
            "source_samples": len(audio), "window_samples": self.loaded.window_samples,
            "starts": list(starts[index]) if self.p4 else [epoch_crop_start(
                len(audio), row["ID"], self.epoch, self.loaded.window_samples)],
            "augmentation": "p4_bag" if self.p4 else (p3_variant(row["ID"], self.epoch)
                if self.p3 else "gain")}, sort_keys=True)
            for index, (row, audio) in enumerate(zip(selected, audios, strict=True)))
        return self._batch(tuple(row["ID"] for row in selected), prepared,
                           [float(row["MUSIC_FAKE"]) for row in selected],
                           [scale * self.weights[row["ID"]] for row in selected], records)

    def _splice(self, index: int) -> Batch:
        if not self.real or not self.fake:
            raise ValueError("P4 needs real and fake music parents")
        base = self._long(self.real, index)
        fake = self._long(self.fake, index)
        control = self._long(self.real, index + 1)
        audio = _audio(self.root / base["path"])
        starts = p4_starts(len(audio), self.loaded.window_samples, base["ID"], self.epoch)
        edited, labels, targets, identifiers = [], [], [], []
        donors = ((fake, True), (control, False)) if not self.p4_control else ((control, False), (base, False))
        for slot, (donor, is_fake) in enumerate(donors):
            value, known, target = p4_splice(audio, _audio(self.root / donor["path"]), starts,
                self.loaded.window_samples, f"{base['ID']}|{donor['ID']}", self.epoch, fake_donor=is_fake)
            edited.append(value)
            labels.append(known)
            targets.append(target)
            identifiers.append(f"p4:{self.epoch}:{index}:{slot}:{base['ID']}|{donor['ID']}")
        prepared = prepare_windows(self.loaded, edited, starts=(starts, starts), window_targets=labels)
        weight = 0.25 / self.splice_pairs
        records = tuple(json.dumps({"ID": identifier, "epoch": self.epoch,
            "parents": identifier.split(":", 4)[-1].split("|"), "starts": list(starts),
            "window_samples": self.loaded.window_samples, "source_samples": len(audio),
            "augmentation": "partial_fake_splice" if target else "matched_real_splice"},
            sort_keys=True) for identifier, target in zip(identifiers, targets, strict=True))
        return self._batch(tuple(identifiers), prepared, targets, (weight, weight), records)

    def _long(self, rows: Sequence[Row], start: int) -> Row:
        for offset in range(len(rows)):
            row = rows[(start + offset) % len(rows)]
            if len(_audio(self.root / row["path"])) >= 5 * 16_000:
                return row
        raise ValueError("P4 has no five-second parent in the requested class")

    @staticmethod
    def _batch(ids: tuple[str, ...], prepared, targets: Sequence[float], weights: Sequence[float],
               records: tuple[str, ...]) -> Batch:
        return Batch(ids, prepared.values, torch.tensor(targets), torch.tensor(weights),
                     prepared.file_indices, prepared.window_targets, prepared.starts, records)


In [ ]:
%%writefile /content/dacon_trainable_v4/v4_training_io.py
"""Atomic artifacts and the one-time frozen G0 prediction cache."""

from __future__ import annotations

import csv
import hashlib
import json
import time
from collections.abc import Mapping, Sequence
from pathlib import Path

from graph_features import Extractor, extract_cached
from graph_fusion import graph_predictions


def digest(path: Path) -> str:
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def atomic_json(path: Path, value: Mapping | Sequence) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False), encoding="utf-8")
    temporary.replace(path)


def append_jsonl(path: Path, value: Mapping) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(value, sort_keys=True, allow_nan=False) + "\n")


def read_predictions(path: Path) -> list[dict[str, str]]:
    with path.open(newline="", encoding="utf-8") as stream:
        return list(csv.DictReader(stream))


def g0_predictions(
    rows: Sequence[Mapping[str, str]],
    data_root: Path,
    baseline: Path,
    output: Path,
    deadline: float,
) -> list[dict[str, str | float]]:
    """Compute complete G0 once, then release its heavyweight models."""
    prediction_path = output / "G0_predictions.csv"
    paths = [data_root / row["path"] for row in rows]
    if any(not path.is_file() for path in paths):
        raise FileNotFoundError("G0 manifest references missing audio")
    dependency_paths = [baseline / "script.py", baseline / "model" / "SHA256SUMS.txt",
                        Path("graph_features.py"), Path("graph_fusion.py"), Path("v2_data.py")]
    if any(not path.is_file() for path in dependency_paths):
        raise FileNotFoundError("G0 dependency receipt cannot be constructed")
    receipt = {"rows": [{"ID": str(row["ID"]), "path": str(row["path"]),
                          "sha256": digest(path)} for row, path in zip(rows, paths, strict=True)],
               "dependencies": {str(path): digest(path) for path in dependency_paths}}
    receipt["cache_key"] = hashlib.sha256(
        json.dumps(receipt, sort_keys=True).encode()).hexdigest()
    receipt_path = output / "g0_receipt.json"
    if (prediction_path.is_file() and (output / "g0_parity.json").is_file()
            and receipt_path.is_file()
            and json.loads(receipt_path.read_text(encoding="utf-8")) == receipt):
        cached = read_predictions(prediction_path)
        if [row["ID"] for row in cached] != [row["ID"] for row in rows]:
            raise ValueError("G0 cache IDs do not match the manifest")
        return cached
    extractor = Extractor(baseline / "script.py", include_raw=False)
    try:
        features = extract_cached(extractor, paths, output / "g0_cache" / receipt["cache_key"][:16], deadline)
        if len(features) != len(rows):
            raise TimeoutError(f"G0 coverage incomplete: {len(features)}/{len(rows)}")
        predictions = []
        for row, feature in zip(rows, features, strict=True):
            feature["ID"] = row["ID"]
            predictions.append(graph_predictions(feature)["G0"])
        parity = []
        for row, prediction in [(row, predictions[index]) for index, row in enumerate(rows)
                                if row.get("panel") == "music_real"][:2]:
            path = data_root / row["path"]
            audio = extractor.base.load_audio(path)
            vp, mp = extractor.base.predict_presence(extractor.panns, extractor.vi, extractor.mi, audio)
            voice, music = extractor.base.separate_voice_and_music(path, extractor.demucs, extractor.device)
            vf = extractor.base.predict_fake(extractor.df, extractor.fake_index, voice, extractor.device)
            mf = extractor.base.predict_fake(extractor.df, extractor.fake_index, music, extractor.device)
            expected = [extractor.base.combine_file_fake_score(vf, mf, vp, mp), vf, mf, vp, mp]
            actual = [float(prediction[name]) for name in ("FILE_FAKE_PROB", "VOICE_FAKE_PROB",
                "MUSIC_FAKE_PROB", "VOICE_PRESENT_PROB", "MUSIC_PRESENT_PROB")]
            delta = max(abs(left - right) for left, right in zip(expected, actual, strict=True))
            if delta > 1e-5:
                raise AssertionError(f"G0 parity failed for {row['ID']}: {delta}")
            parity.append({"ID": row["ID"], "max_probability_delta": delta})
        atomic_json(output / "g0_parity.json", parity)
    finally:
        extractor.close()
    with prediction_path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=predictions[0].keys())
        writer.writeheader()
        writer.writerows(predictions)
    atomic_json(receipt_path, receipt)
    return predictions


def budget_event(path: Path, event: str, **fields: str | int | float | bool) -> None:
    append_jsonl(path, {"event": event, "unix": time.time(), **fields})


In [ ]:
%%writefile /content/dacon_trainable_v4/v4_batches.py
"""Deterministic full-source crop scheduling for V4 training."""

from __future__ import annotations

import hashlib
import math
import random
import subprocess
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Final

import numpy as np
import imageio_ffmpeg

SEED: Final = 20_260_913
STRATA: Final = 8


def _seed(*parts: str | int) -> int:
    text = "|".join(map(str, (SEED, *parts)))
    return int(hashlib.sha256(text.encode()).hexdigest()[:16], 16)


def epoch_crop(
    audio: np.ndarray,
    identifier: str,
    epoch: int,
    window_samples: int,
) -> np.ndarray:
    """Draw one replayable crop while cycling all eight temporal strata."""
    waveform = np.asarray(audio, dtype=np.float32)
    if waveform.ndim != 1 or waveform.size == 0:
        raise ValueError("training audio must be nonempty mono samples")
    if window_samples <= 0:
        raise ValueError("window_samples must be positive")
    if waveform.size <= window_samples:
        return np.pad(waveform, (0, window_samples - waveform.size))
    start = epoch_crop_start(waveform.size, identifier, epoch, window_samples)
    return waveform[start : start + window_samples].copy()


def epoch_crop_start(samples: int, identifier: str, epoch: int, window_samples: int) -> int:
    if samples <= window_samples:
        return 0
    positions = samples - window_samples + 1
    strata = min(STRATA, positions)
    stratum = (_seed(identifier) + epoch) % strata
    lower = positions * stratum // strata
    upper = positions * (stratum + 1) // strata
    start = random.Random(_seed(identifier, epoch, "crop")).randrange(lower, upper)
    return start


def apply_gain(audio: np.ndarray, identifier: str, epoch: int) -> np.ndarray:
    gain_db = random.Random(_seed(identifier, epoch, "gain")).uniform(-6.0, 6.0)
    return (audio * 10 ** (gain_db / 20.0)).astype(np.float32)


def epoch_order(rows: Sequence[Mapping[str, str]], epoch: int) -> list[Mapping[str, str]]:
    ordered = list(rows)
    random.Random(_seed(epoch, "order")).shuffle(ordered)
    return ordered


def music_supervision_rows(rows: Sequence[Mapping[str, str]]) -> list[Mapping[str, str]]:
    """Keep only rows with justified music-present authenticity supervision."""
    return [row for row in rows if row.get("MUSIC_PRESENT") == "1" and row.get("MUSIC_FAKE") in {"0", "1"}]


def p4_starts(samples: int, window_samples: int, identifier: str, epoch: int) -> tuple[int, ...]:
    """Spread up to four distinct native windows across the complete source."""
    if samples <= window_samples:
        return (0,)
    positions = samples - window_samples + 1
    if positions <= 4:
        return tuple(range(positions))
    starts: list[int] = []
    for stratum in range(4):
        lower = positions * stratum // 4
        upper = positions * (stratum + 1) // 4
        starts.append(random.Random(_seed(identifier, epoch, stratum, "bag")).randrange(lower, upper))
    return tuple(starts)


def p4_splice(
    real: np.ndarray,
    donor: np.ndarray,
    starts: Sequence[int],
    window_samples: int,
    identifier: str,
    epoch: int,
    *,
    fake_donor: bool,
) -> tuple[np.ndarray, tuple[float, ...], float]:
    """Make one aligned real/fake or matched real/real partial edit."""
    samples = min(len(real), len(donor))
    if samples < 5 * 16_000:
        raise ValueError("P4 splice parents must both contain at least five seconds")
    duration = min(samples, random.Random(_seed(identifier, epoch, "splice-duration")).randint(5, 20) * 16_000)
    offset = random.Random(_seed(identifier, epoch, "splice-offset")).randrange(samples - duration + 1)
    output = real.copy()
    output[offset : offset + duration] = donor[offset : offset + duration]
    crossfade = min(800, duration // 2)
    for boundary, entering in ((offset, True), (offset + duration, False)):
        lower, upper = max(0, boundary - crossfade), min(samples, boundary + crossfade)
        if upper > lower:
            donor_weight = np.linspace(0.0, 1.0, upper - lower, dtype=np.float32)
            if not entering:
                donor_weight = 1.0 - donor_weight
            output[lower:upper] = (real[lower:upper] * (1.0 - donor_weight)
                                   + donor[lower:upper] * donor_weight)
    labels: list[float] = []
    for start in starts:
        end = start + window_samples
        wholly_replaced = start >= offset + crossfade and end <= offset + duration - crossfade
        wholly_real = end <= offset - crossfade or start >= offset + duration + crossfade
        labels.append(float(fake_donor) if wholly_replaced else (0.0 if wholly_real else math.nan))
    retained_fake = any(max(start, offset) < min(start + window_samples, offset + duration)
                        for start in starts)
    bag_target = float(fake_donor and retained_fake)
    return output, tuple(labels), bag_target


def _aligned(audio: np.ndarray, samples: int) -> np.ndarray:
    return np.pad(audio[:samples], (0, max(0, samples - len(audio))))


def p3_augment(
    audio: np.ndarray,
    identifier: str,
    epoch: int,
    cache_root: str,
) -> np.ndarray:
    """Apply the fixed, label-independent P3 choice and cache codec work on CPU."""
    choice = random.Random(_seed(identifier, epoch, "p3-choice")).random()
    if choice < 0.5:
        return audio.copy()
    if choice < 0.8:
        codec_index = _seed(identifier, epoch, "codec") % 6
        codec = "libmp3lame" if codec_index < 3 else "aac"
        bitrate = (64, 128, 192)[codec_index % 3]
        key = hashlib.sha256(audio.astype("<f4", copy=False).tobytes()).hexdigest()
        cache = Path(cache_root) / f"{key}-{codec}-{bitrate}.npy"
        if cache.is_file():
            return np.load(cache)
        cache.parent.mkdir(parents=True, exist_ok=True)
        ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()
        encoded = subprocess.run(
            [ffmpeg, "-v", "error", "-f", "f32le", "-ar", "16000", "-ac", "1",
             "-i", "-", "-f", "mp3" if codec == "libmp3lame" else "adts",
             "-c:a", codec, "-b:a", f"{bitrate}k", "-"],
            input=audio.astype("<f4", copy=False).tobytes(), check=True, capture_output=True,
        ).stdout
        decoded = subprocess.run(
            [ffmpeg, "-v", "error", "-i", "-", "-f", "f32le", "-ar", "16000",
             "-ac", "1", "-"], input=encoded, check=True, capture_output=True,
        ).stdout
        result = _aligned(np.frombuffer(decoded, dtype="<f4").copy(), len(audio))
        temporary = cache.with_suffix(".tmp.npy")
        np.save(temporary, result)
        temporary.replace(cache)
        return result
    if choice < 0.9:
        down = np.interp(
            np.linspace(0, len(audio) - 1, round(len(audio) * 0.75)),
            np.arange(len(audio)), audio,
        )
        return np.interp(
            np.linspace(0, len(down) - 1, len(audio)), np.arange(len(down)), down,
        ).astype(np.float32)
    rng = np.random.default_rng(_seed(identifier, epoch, "noise"))
    snr_db = random.Random(_seed(identifier, epoch, "snr")).uniform(20.0, 35.0)
    rms = max(float(np.sqrt(np.mean(np.square(audio), dtype=np.float64))), 1e-8)
    noise = rng.standard_normal(len(audio)).astype(np.float32)
    noise *= rms / (max(float(np.sqrt(np.mean(np.square(noise), dtype=np.float64))), 1e-8)
                    * 10 ** (snr_db / 20.0))
    return audio + noise


def p3_variant(identifier: str, epoch: int) -> str:
    choice = random.Random(_seed(identifier, epoch, "p3-choice")).random()
    if choice < 0.5:
        return "identity"
    if choice < 0.8:
        codec_index = _seed(identifier, epoch, "codec") % 6
        return f"{'mp3' if codec_index < 3 else 'aac'}-{(64, 128, 192)[codec_index % 3]}k"
    return "resample-16k-12k-16k" if choice < 0.9 else "noise-20-35db"


In [ ]:
%%writefile /content/dacon_trainable_v4/v4_model.py
"""Full-fine-tuning adapters for the pinned V4 music encoders."""

from __future__ import annotations

import math
from collections.abc import Sequence
from pathlib import Path
from typing import Final, assert_never

import numpy as np
import torch
from safetensors.torch import load_file
from torch.nn import functional

from recovery_sonics import SonicsPredictor
from v4_model_components import (
    DisabledAugment,
    FilePredictions,
    LoadedModel,
    ModelKind,
    ParameterGroups,
    PerExampleAmplitudeToDB,
    PreparedWindows,
    V4ModelError,
    load_ast,
)

SONICS_SAMPLES: Final = 80_000
AST_SAMPLES: Final = 160_000
AST_KALDI_MIN_SAMPLES: Final = 400


def load_model(
    kind: ModelKind,
    assets: Path,
    device: str | torch.device,
    *,
    state_path: Path | None = None,
) -> LoadedModel:
    """Load a verified original, then optionally overlay one strict trained state."""
    match kind:
        case "sonics":
            model = SonicsPredictor(assets, str(device)).model
            feature_extractor = model.ft_extractor
            feature_extractor.amplitude_to_db = PerExampleAmplitudeToDB(
                feature_extractor.amplitude_to_db,
            )
            model.augment = DisabledAugment()
            processor = None
            window_samples = SONICS_SAMPLES
        case "ast":
            model, processor = load_ast(assets, device)
            window_samples = AST_SAMPLES
        case unreachable:
            assert_never(unreachable)
    if state_path is not None:
        model.load_state_dict(load_file(state_path, device="cpu"), strict=True)
    model.requires_grad_(True)
    return LoadedModel(kind, model, processor, 16_000, window_samples)


def parameter_groups(loaded: LoadedModel) -> ParameterGroups:
    """Separate the existing/new classifier from every encoder parameter."""
    head_prefix = "classifier." if loaded.kind == "sonics" else "head."
    named = tuple(loaded.model.named_parameters())
    return ParameterGroups(
        encoder=tuple(item for item in named if not item[0].startswith(head_prefix)),
        head=tuple(item for item in named if item[0].startswith(head_prefix)),
    )


def _validated_audio(audio: np.ndarray) -> np.ndarray:
    waveform = np.asarray(audio)
    if waveform.ndim != 1 or waveform.size == 0:
        raise V4ModelError("Audio must be nonempty and one-dimensional")
    try:
        waveform = np.ascontiguousarray(waveform, dtype=np.float32)
    except (TypeError, ValueError) as error:
        raise V4ModelError("Audio must be convertible to float32") from error
    if not np.isfinite(waveform).all():
        raise V4ModelError("Audio must contain only finite values")
    return waveform


def _native_starts(samples: int, kind: ModelKind) -> tuple[int, ...]:
    match kind:
        case "sonics":
            return tuple(SonicsPredictor._window_starts(samples))
        case "ast":
            return tuple(range(0, samples, AST_SAMPLES))
        case unreachable:
            assert_never(unreachable)


def prepare_windows(
    loaded: LoadedModel,
    audios: Sequence[np.ndarray],
    *,
    starts: Sequence[Sequence[int]] | None = None,
    window_targets: Sequence[Sequence[float]] | None = None,
) -> PreparedWindows:
    """Prepare each file on its native deterministic grid without cross-file state."""
    if not audios:
        raise V4ModelError("At least one audio file is required")
    waveforms = tuple(_validated_audio(audio) for audio in audios)
    if starts is None:
        resolved_starts = tuple(
            _native_starts(audio.size, loaded.kind) for audio in waveforms
        )
    else:
        if len(starts) != len(waveforms):
            raise V4ModelError("Explicit starts must contain one grid per file")
        resolved_starts = tuple(tuple(file_starts) for file_starts in starts)
        for audio, file_starts in zip(waveforms, resolved_starts, strict=True):
            last_full_start = max(0, audio.size - loaded.window_samples)
            if (
                not 1 <= len(file_starts) <= 4
                or len(set(file_starts)) != len(file_starts)
                or any(start < 0 or start > last_full_start for start in file_starts)
            ):
                raise V4ModelError(
                    "Explicit starts must be 1-4 distinct full-window starts"
                )
    if window_targets is not None and len(window_targets) != len(waveforms):
        raise V4ModelError("Window labels must contain one grid per file")
    windows: list[np.ndarray] = []
    indices: list[int] = []
    labels: list[float] = []
    for file_index, (audio, file_starts) in enumerate(
        zip(waveforms, resolved_starts, strict=True)
    ):
        supplied = None if window_targets is None else tuple(window_targets[file_index])
        if supplied is not None and len(supplied) != len(file_starts):
            raise V4ModelError("Window labels must match the native window grid")
        for window_index, start in enumerate(file_starts):
            window = audio[start : start + loaded.window_samples]
            windows.append(window)
            indices.append(file_index)
            labels.append(math.nan if supplied is None else supplied[window_index])
    if loaded.kind == "sonics":
        values = np.zeros((len(windows), loaded.window_samples), dtype=np.float32)
        for index, window in enumerate(windows):
            values[index, : window.size] = window
        tensor = torch.from_numpy(values)
    else:
        if loaded.processor is None:
            raise V4ModelError("AST processor is unavailable")
        processor_windows = [
            np.pad(window, (0, max(0, AST_KALDI_MIN_SAMPLES - window.size)))
            for window in windows
        ]
        tensor = loaded.processor(
            processor_windows,
            sampling_rate=loaded.sample_rate,
            return_tensors="pt",
        )["input_values"].float()
    return PreparedWindows(
        tensor,
        torch.tensor(indices, dtype=torch.long),
        resolved_starts,
        torch.tensor(labels, dtype=torch.float32),
    )


def window_logits(loaded: LoadedModel, prepared: PreparedWindows) -> torch.Tensor:
    """Return one fake-music logit for every prepared native window."""
    device = next(loaded.model.parameters()).device
    logits = loaded.model(prepared.values.to(device))
    if logits.ndim == 2 and logits.shape[1] == 1:
        logits = logits[:, 0]
    if logits.ndim != 1 or logits.shape[0] != prepared.file_indices.shape[0]:
        raise V4ModelError("Model must return one logit per native window")
    return logits


def pool_bag_logits(
    logits: torch.Tensor,
    file_indices: torch.Tensor,
    file_count: int,
    temperature: float = 2.0,
) -> torch.Tensor:
    """Apply fixed normalized log-sum-exp independently within each file."""
    if (
        logits.ndim != 1
        or file_indices.shape != logits.shape
        or file_count < 1
        or temperature <= 0
    ):
        raise V4ModelError("Bag pooling input contract mismatch")
    pooled: list[torch.Tensor] = []
    for file_index in range(file_count):
        selected = logits[file_indices.to(logits.device) == file_index]
        if selected.numel() == 0:
            raise V4ModelError("Every file must contain at least one window")
        pooled.append(
            torch.logsumexp(temperature * selected, dim=0) / temperature
            - math.log(selected.numel()) / temperature
        )
    return torch.stack(pooled)


def _known_binary(values: torch.Tensor, name: str) -> torch.Tensor:
    known = torch.isfinite(values)
    if torch.any(known & (values != 0) & (values != 1)):
        raise V4ModelError(f"{name} must contain binary values or NaN")
    return known


def music_loss(
    logits: torch.Tensor,
    prepared: PreparedWindows,
    targets: torch.Tensor,
    *,
    p4: bool,
) -> torch.Tensor:
    """Compute ordinary window BCE or P4 bag BCE plus known-segment BCE."""
    if targets.ndim != 1 or len(targets) != len(prepared.starts):
        raise V4ModelError("Music targets must contain one value per file")
    targets = targets.to(logits.device)
    known_bags = _known_binary(targets, "Music targets")
    if not p4:
        window_targets = targets[prepared.file_indices.to(logits.device)]
        known_windows = torch.isfinite(window_targets)
        if not torch.any(known_windows):
            raise V4ModelError("Batch has no known music labels")
        return functional.binary_cross_entropy_with_logits(
            logits[known_windows], window_targets[known_windows]
        )
    losses: list[torch.Tensor] = []
    pooled = pool_bag_logits(logits, prepared.file_indices, len(targets))
    if torch.any(known_bags):
        losses.append(
            functional.binary_cross_entropy_with_logits(
                pooled[known_bags], targets[known_bags]
            )
        )
    segment_targets = prepared.window_targets.to(logits.device)
    known_segments = _known_binary(segment_targets, "Window targets")
    if torch.any(known_segments):
        losses.append(
            0.5
            * functional.binary_cross_entropy_with_logits(
                logits[known_segments],
                segment_targets[known_segments],
            )
        )
    if not losses:
        raise V4ModelError("Batch has no known bag or segment labels")
    return torch.stack(losses).sum()


def predict_files(
    loaded: LoadedModel, audios: Sequence[np.ndarray], *, p4: bool
) -> FilePredictions:
    """Predict full files with their native grid and fixed requested pooling."""
    prepared = prepare_windows(loaded, audios)
    with torch.inference_mode():
        windows = window_logits(loaded, prepared)
        if p4:
            logits = pool_bag_logits(windows, prepared.file_indices, len(audios))
            probabilities = torch.sigmoid(logits)
        else:
            probabilities = torch.stack(
                [
                    torch.sigmoid(
                        windows[prepared.file_indices.to(windows.device) == index]
                    ).mean()
                    for index in range(len(audios))
                ]
            )
            logits = torch.logit(probabilities)
    return FilePredictions(logits.cpu(), probabilities.cpu(), prepared.starts)


In [ ]:
%%writefile /content/dacon_trainable_v4/v4_model_components.py
"""Small model-specific components shared by the V4 adapter."""

from __future__ import annotations

import hashlib
from dataclasses import dataclass, replace
from pathlib import Path
from typing import Final, Literal

import torch
from torch import nn
from transformers import ASTFeatureExtractor, ASTModel

from ast_multitask import load_processor

AST_HASHES: Final = {
    "config.json": "a93d525511d77e8ecc933d09674b85099815bbbb417c228a4edd655e252fb9ff",
    "preprocessor_config.json": "8d04ba5a9c6fca5d39d0de2b1fd05ecf79deb589fbba279728bbebac39934231",
    "model.safetensors": "ae0c1e2ad4e1381d851fa9bf298ba13ebc9c5a914cdee2dbe427a6583869924d",
}
ModelKind = Literal["sonics", "ast"]


class V4ModelError(ValueError):
    """A V4 model asset, waveform, or label violates the fixed contract."""


@dataclass(frozen=True, slots=True)
class LoadedModel:
    kind: ModelKind
    model: nn.Module
    processor: ASTFeatureExtractor | None
    sample_rate: int
    window_samples: int


@dataclass(frozen=True, slots=True)
class PreparedWindows:
    values: torch.Tensor
    file_indices: torch.Tensor
    starts: tuple[tuple[int, ...], ...]
    window_targets: torch.Tensor

    def to(self, device: str | torch.device) -> PreparedWindows:
        return replace(
            self,
            values=self.values.to(device),
            file_indices=self.file_indices.to(device),
            window_targets=self.window_targets.to(device),
        )


@dataclass(frozen=True, slots=True)
class ParameterGroups:
    encoder: tuple[tuple[str, nn.Parameter], ...]
    head: tuple[tuple[str, nn.Parameter], ...]


@dataclass(frozen=True, slots=True)
class FilePredictions:
    logits: torch.Tensor
    probabilities: torch.Tensor
    starts: tuple[tuple[int, ...], ...]


class ASTMusicModel(nn.Module):
    def __init__(self, backbone: ASTModel) -> None:
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(backbone.config.hidden_size, 1)

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        return self.head(
            self.backbone(input_values=input_values).pooler_output
        ).squeeze(-1)


class PerExampleAmplitudeToDB(nn.Module):
    """Give torchaudio a channel axis so top_db is computed per example."""

    def __init__(self, transform: nn.Module) -> None:
        super().__init__()
        self.transform = transform

    def forward(self, spectrogram: torch.Tensor) -> torch.Tensor:
        return self.transform(spectrogram.unsqueeze(1)).squeeze(1)


class DisabledAugment(nn.Module):
    def forward(
        self,
        spectrogram: torch.Tensor,
        targets: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        return spectrogram, targets


def load_ast(
    assets: Path, device: str | torch.device
) -> tuple[nn.Module, ASTFeatureExtractor]:
    """Verify and load the pinned AST backbone plus one new music head."""
    for name, expected in AST_HASHES.items():
        path = assets / name
        if not path.is_file():
            raise V4ModelError(f"Pinned AST asset is missing: {path}")
        with path.open("rb") as stream:
            actual = hashlib.file_digest(stream, "sha256").hexdigest()
        if actual != expected:
            raise V4ModelError(f"Pinned AST asset hash mismatch: {name}")
    processor = load_processor(assets)
    backbone = ASTModel.from_pretrained(
        assets,
        local_files_only=True,
        attn_implementation="sdpa",
    )
    return ASTMusicModel(backbone).to(device), processor


In [ ]:
%%writefile /content/dacon_trainable_v4/v4_data.py
"""Runtime loaders for the fixed V4 manifests and full-source audio."""
from __future__ import annotations

import csv
from pathlib import Path

import numpy as np

from v2_data import decode_audio


def load_manifest(data_root: Path, split: str) -> list[dict[str, str]]:
    path = data_root / f"{split}.csv"
    with path.open(newline="") as handle:
        return list(csv.DictReader(handle))


def load_audio(path: Path) -> np.ndarray:
    return decode_audio(path, limit_seconds=None)


In [ ]:
%%writefile /content/dacon_trainable_v4/v4_evaluation.py
"""Exact V4 metrics on fixed development and sealed panels."""

from __future__ import annotations

import random
from collections.abc import Mapping, Sequence
from math import isclose
from typing import Final

from graph_scoring import _roc_metrics, official_metrics, validate_predictions
from v2_scoring import _components

Scalar = str | int | float | bool | None
Row = Mapping[str, str | int | float | None]
Report = dict[str, Scalar | dict | list]
PANEL_COUNTS: Final = {
    "music_real": 10, "music_seen": 30, "music_ood": 10,
    "speech": 40, "mixture": 80, "splice": 40,
}
PROTECTED: Final = ("VOICE_FAKE_PROB", "VOICE_PRESENT_PROB", "MUSIC_PRESENT_PROB")
FILE_FORMULA_ATOL: Final = 2e-10


def _label(row: Row, name: str) -> int | None:
    value = row.get(name)
    if value in (None, ""):
        return None
    if value not in (0, 1, "0", "1"):
        raise ValueError(f"invalid binary label: {name}")
    return int(value)


def _prediction_maps(rows: Sequence[Row], baseline: Sequence[Row], candidate: Sequence[Row]) -> tuple[dict[str, Row], dict[str, Row]]:
    expected = [str(row["ID"]) for row in rows]
    parsed_base = validate_predictions(baseline, expected)
    parsed_candidate = validate_predictions(candidate, expected)
    base_map: dict[str, Row] = {row["ID"]: row for row in parsed_base}
    candidate_map: dict[str, Row] = {row["ID"]: row for row in parsed_candidate}
    for identifier in expected:
        base, changed = base_map[identifier], candidate_map[identifier]
        if any(changed[column] != base[column] for column in PROTECTED):
            raise ValueError(f"protected G0 output changed: {identifier}")
        expected_file = max(
            float(base["VOICE_PRESENT_PROB"]) * float(base["VOICE_FAKE_PROB"]),
            float(base["MUSIC_PRESENT_PROB"]) * float(changed["MUSIC_FAKE_PROB"]),
        )
        if not isclose(float(changed["FILE_FAKE_PROB"]), expected_file,
                       rel_tol=0.0, abs_tol=FILE_FORMULA_ATOL):
            raise ValueError(f"FILE formula mismatch: {identifier}")
    return base_map, candidate_map


def _head(rows: Sequence[Row], predictions: Mapping[str, Row], target: str,
          score: str, mask: str | None = None) -> dict[str, float | int | None]:
    targets: list[int] = []
    scores: list[float] = []
    unknown = 0
    for row in rows:
        if mask is not None:
            present = _label(row, mask)
            if present is None:
                unknown += 1
                continue
            if present == 0:
                continue
        truth = _label(row, target)
        if truth is None:
            unknown += 1
            continue
        targets.append(truth)
        scores.append(float(predictions[str(row["ID"])][score]))
    if len(set(targets)) != 2:
        return {"eer": None, "auc": None, "n": len(targets), "unknown": unknown}
    eer, auc = _roc_metrics(targets, scores)
    return {"eer": eer, "auc": auc, "n": len(targets), "unknown": unknown}


def _comparison(rows: Sequence[Row], baseline: Mapping[str, Row], candidate: Mapping[str, Row]) -> Report:
    base_file = _head(rows, baseline, "FILE_FAKE", "FILE_FAKE_PROB")
    changed_file = _head(rows, candidate, "FILE_FAKE", "FILE_FAKE_PROB")
    base_music = _head(rows, baseline, "MUSIC_FAKE", "MUSIC_FAKE_PROB", "MUSIC_PRESENT")
    changed_music = _head(rows, candidate, "MUSIC_FAKE", "MUSIC_FAKE_PROB", "MUSIC_PRESENT")
    values = (base_file["eer"], changed_file["eer"], base_music["eer"], changed_music["eer"])
    gain = None if any(value is None for value in values) else (
        .45 * (float(base_file["eer"]) - float(changed_file["eer"]))
        + .27 * (float(base_music["eer"]) - float(changed_music["eer"]))
    )
    return {"D": gain, "G0": {"file": base_file, "music": base_music},
            "candidate": {"file": changed_file, "music": changed_music}}


def _class_support(rows: Sequence[Row]) -> dict[str, int]:
    support = {"0": 0, "1": 0}
    for unit in _components(rows):
        labels = {_label(rows[index], "MUSIC_FAKE") for index in unit
                  if _label(rows[index], "MUSIC_PRESENT") == 1}
        for label in (0, 1):
            support[str(label)] += int(label in labels)
    return support


def _bootstrap(rows: Sequence[Row], baseline: Mapping[str, Row], candidate: Mapping[str, Row]) -> Report:
    units = _components(rows)
    rng = random.Random(20260913)
    deltas: list[float] = []
    for _ in range(2000):
        sampled = [rows[index] for unit in (rng.choice(units) for _ in units) for index in unit]
        delta = _comparison(sampled, baseline, candidate)["D"]
        if isinstance(delta, float):
            deltas.append(delta)
    deltas.sort()
    return {"seed": 20260913, "resamples": 2000, "lineage_components": len(units),
            "class_group_support": _class_support(rows), "valid_resamples": len(deltas),
            "valid_rate": len(deltas) / 2000,
            "win_rate": None if not deltas else sum(value > 0 for value in deltas) / len(deltas),
            "p10_delta": None if not deltas else deltas[(len(deltas) - 1) // 10]}


def _validate_panel(rows: Sequence[Row], split: str) -> None:
    counts = {panel: 0 for panel in PANEL_COUNTS}
    for row in rows:
        if row.get("split") != split:
            raise ValueError(f"evaluation rows must use split={split}")
        panel = row.get("panel")
        if panel not in counts:
            raise ValueError(f"unknown evaluation panel: {panel}")
        counts[str(panel)] += 1
    if counts != PANEL_COUNTS:
        raise ValueError(f"evaluation panel counts must be exactly {PANEL_COUNTS}")


def _evaluate(rows: Sequence[Row], baseline: Sequence[Row], candidate: Sequence[Row], split: str) -> Report:
    _validate_panel(rows, split)
    base_map, candidate_map = _prediction_maps(rows, baseline, candidate)
    real = [row for row in rows if row["panel"] == "music_real"]
    music_panels: dict[str, Report] = {}
    for panel in ("music_seen", "music_ood"):
        selected = [*real, *(row for row in rows if row["panel"] == panel)]
        comparison = _comparison(selected, base_map, candidate_map)
        comparison["bootstrap"] = _bootstrap(selected, base_map, candidate_map)
        music_panels[panel] = comparison
    controlled = {panel: _comparison(
        [row for row in rows if row["panel"] == panel], base_map, candidate_map)
        for panel in ("speech", "mixture", "splice")}
    absolute = official_metrics(rows, [candidate_map[str(row["ID"])] for row in rows])
    gains = [music_panels[panel]["D"] for panel in music_panels]
    aucs = [music_panels[panel]["candidate"]["music"]["auc"] for panel in music_panels]
    return {"split": split, "absolute": absolute, "music": music_panels,
            "controlled": controlled,
            "rank_score": None if any(value is None for value in gains) else min(gains),
            "music_auc_tiebreak": None if any(value is None for value in aucs) else min(aucs)}


def evaluate_development(rows: Sequence[Row], baseline: Sequence[Row], candidate: Sequence[Row]) -> Report:
    return _evaluate(rows, baseline, candidate, "dev")


def evaluate_sealed(rows: Sequence[Row], baseline: Sequence[Row], candidate: Sequence[Row]) -> Report:
    return _evaluate(rows, baseline, candidate, "sealed")


In [ ]:
%%writefile /content/dacon_trainable_v4/v4_selection.py
"""V4 development selection and one-way sealed acceptance."""

from __future__ import annotations

import csv
import hashlib
import json
import time
from collections.abc import Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Final

from v4_evaluation import Report, Row, evaluate_development, evaluate_sealed

PIPELINES: Final = ("P1", "P2", "P3", "P4")
RECIPE_FIELDS: Final = {
    "selected", "parent", "treatment", "learning_rate", "scheduler", "epoch_count",
    "stage_epochs", "data_hashes", "source_hashes", "pooling", "seeds", "gate_thresholds",
}


@dataclass(frozen=True, slots=True)
class SelectionError(ValueError):
    detail: str

    def __str__(self) -> str:
        return self.detail


def _digest(path: Path) -> str:
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def _write_immutable(path: Path, payload: Mapping[str, str | int | float | bool | dict | list]) -> str:
    serialized = json.dumps(payload, sort_keys=True, indent=2, allow_nan=False) + "\n"
    if path.is_file():
        if path.read_text(encoding="utf-8") != serialized:
            raise SelectionError(f"immutable lock changed: {path.name}")
        return _digest(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(serialized, encoding="utf-8")
    temporary.replace(path)
    return _digest(path)


def _write_timestamped(path: Path, payload: Mapping[str, str | int | float | bool | dict | list],
                       timestamp_key: str) -> str:
    if path.is_file():
        existing = json.loads(path.read_text(encoding="utf-8"))
        existing.pop(timestamp_key, None)
        if existing != dict(payload):
            raise SelectionError(f"immutable lock changed: {path.name}")
        return _digest(path)
    return _write_immutable(path, {**payload, timestamp_key: time.time()})


def _quality_gates(report: Report) -> dict[str, bool]:
    music = report["music"]
    controlled = report["controlled"]
    gates: dict[str, bool] = {}
    for panel, label in (("music_seen", "seen"), ("music_ood", "ood")):
        result = music[panel]
        base, candidate = result["G0"], result["candidate"]
        bootstrap = result["bootstrap"]
        gates[f"{label}_gain"] = result["D"] is not None and result["D"] >= .005
        for head in ("file", "music"):
            gates[f"{label}_{head}_eer"] = (
                candidate[head]["eer"] is not None and base[head]["eer"] is not None
                and candidate[head]["eer"] <= base[head]["eer"] + .01)
        gates[f"{label}_music_auc"] = (
            candidate["music"]["auc"] is not None and base["music"]["auc"] is not None
            and candidate["music"]["auc"] >= base["music"]["auc"] - .01)
        support = bootstrap["class_group_support"]
        gates[f"{label}_class_support"] = support["0"] >= 10 and support["1"] >= 10
        gates[f"{label}_bootstrap_valid"] = bootstrap["valid_rate"] >= .90
        gates[f"{label}_bootstrap_win"] = (
            bootstrap["win_rate"] is not None and bootstrap["win_rate"] >= .80)
        gates[f"{label}_bootstrap_p10"] = (
            bootstrap["p10_delta"] is not None and bootstrap["p10_delta"] >= 0)
    for panel in ("speech", "mixture"):
        base = controlled[panel]["G0"]["file"]["eer"]
        candidate = controlled[panel]["candidate"]["file"]["eer"]
        gates[f"{panel}_file_eer"] = (
            base is not None and candidate is not None and candidate <= base + .02)
    return gates


def select_development(rows: Sequence[Row], baseline: Sequence[Row],
                       candidates: Mapping[str, Sequence[Row]],
                       runtime_seconds: Mapping[str, float],
                       projected_refit_seconds: Mapping[str, float],
                       reserved_refit_seconds: float) -> Report:
    unknown = set(candidates) - set(PIPELINES)
    if unknown:
        raise SelectionError(f"unknown V4 pipelines: {sorted(unknown)}")
    reports: dict[str, dict] = {}
    eligible: list[str] = []
    for name in PIPELINES:
        if name not in candidates:
            reports[name] = {"status": "not_completed", "gates": {"passed": False}}
            continue
        if name not in runtime_seconds or name not in projected_refit_seconds:
            raise SelectionError(f"missing runtime forecast: {name}")
        evaluation = evaluate_development(rows, baseline, candidates[name])
        gates = _quality_gates(evaluation)
        gates["resource"] = 1.25 * projected_refit_seconds[name] <= reserved_refit_seconds
        gates["passed"] = all(gates.values())
        reports[name] = {"status": "completed", "evaluation": evaluation, "gates": gates,
                         "runtime_seconds": runtime_seconds[name],
                         "projected_refit_seconds": projected_refit_seconds[name]}
        if gates["passed"]:
            eligible.append(name)
    selected = max(eligible, key=lambda name: (
        reports[name]["evaluation"]["rank_score"],
        reports[name]["evaluation"]["music_auc_tiebreak"],
        -runtime_seconds[name], -PIPELINES.index(name)), default="G0")
    return {"selected": selected,
            "reason": "eligible_development_winner" if selected != "G0" else "no_candidate_passed",
            "candidate_order": list(PIPELINES), "candidates": reports}


def _read_ids(path: Path) -> list[str]:
    with path.open(encoding="utf-8-sig", newline="") as stream:
        rows = list(csv.DictReader(stream))
    identifiers = [str(row.get("ID", "")) for row in rows]
    if not identifiers or any(not identifier for identifier in identifiers) or len(set(identifiers)) != len(identifiers):
        raise SelectionError(f"manifest IDs are incomplete: {path.name}")
    return identifiers


def _verify_recipe_inputs(recipe: Mapping[str, str | int | float | bool | dict | list]) -> list[str]:
    for name in ("selection", "eligible_manifest"):
        target = Path(str(recipe[name]))
        if not target.is_file() or _digest(target) != recipe[f"{name}_sha256"]:
            raise SelectionError(f"{name} hash changed after recipe lock")
    return _read_ids(Path(str(recipe["eligible_manifest"])))


def write_recipe_lock(path: Path, selection_path: Path, eligible_manifest: Path,
                      payload: Mapping[str, str | int | float | bool | dict | list]) -> str:
    selected = payload.get("selected")
    if selected not in PIPELINES:
        raise SelectionError("recipe lock requires one selected V4 pipeline")
    missing = RECIPE_FIELDS - set(payload)
    if missing:
        raise SelectionError(f"recipe lock fields are missing: {sorted(missing)}")
    if not selection_path.is_file() or not eligible_manifest.is_file():
        raise SelectionError("recipe lock input is missing")
    selection = json.loads(selection_path.read_text(encoding="utf-8"))
    selected_report = selection.get("candidates", {}).get(selected, {})
    gates = selected_report.get("gates", {})
    if selection.get("selected") != selected or selected_report.get("status") != "completed":
        raise SelectionError("recipe does not match the development winner")
    if gates.get("passed") is not True or gates.get("resource") is not True:
        raise SelectionError("development winner is not eligible for refit")
    stage_epochs = payload["stage_epochs"]
    if not isinstance(stage_epochs, dict) or not stage_epochs or any(
            type(value) is not int or value <= 0 for value in stage_epochs.values()):
        raise SelectionError("recipe stage epochs are invalid")
    if sum(stage_epochs.values()) != payload["epoch_count"]:
        raise SelectionError("recipe epoch count does not match its stages")
    _read_ids(eligible_manifest)
    locked = {**payload, "selection": str(selection_path),
              "selection_sha256": _digest(selection_path),
              "eligible_manifest": str(eligible_manifest),
              "eligible_manifest_sha256": _digest(eligible_manifest)}
    return _write_timestamped(path, locked, "locked_unix")


def _verify_refit_manifest(path: Path, expected_ids: Sequence[str], expected_exposures: int) -> None:
    with path.open(encoding="utf-8-sig", newline="") as stream:
        rows = list(csv.DictReader(stream))
    required = {"ID", "expected_exposures", "actual_exposures"}
    if not rows or set(rows[0]) < required or [row["ID"] for row in rows] != list(expected_ids):
        raise SelectionError("refit manifest is incomplete")
    try:
        complete = all(int(row["expected_exposures"]) == expected_exposures
                       and int(row["actual_exposures"]) == expected_exposures
                       for row in rows)
    except ValueError as error:
        raise SelectionError("refit manifest has invalid exposure counts") from error
    if not complete:
        raise SelectionError("refit exposures are incomplete")


def write_weight_lock(path: Path, recipe_lock: Path, weights: Path,
                      inference_code: Sequence[Path], refit_manifest: Path,
                      refit_report: Mapping[str, bool | int | float | str]) -> str:
    if not refit_report.get("completed") or not refit_report.get("exposures_complete"):
        raise SelectionError("refit is incomplete")
    inputs = (recipe_lock, weights, refit_manifest, *inference_code)
    if not all(item.is_file() for item in inputs):
        raise SelectionError("weight lock input is missing")
    recipe = json.loads(recipe_lock.read_text(encoding="utf-8"))
    eligible_ids = _verify_recipe_inputs(recipe)
    started = refit_report.get("started_unix")
    completed = refit_report.get("completed_unix")
    prescribed_epochs = refit_report.get("prescribed_epochs")
    completed_epochs = refit_report.get("completed_epochs")
    if type(started) not in (int, float) or type(completed) not in (int, float):
        raise SelectionError("refit chronology is incomplete")
    if not recipe["locked_unix"] <= started <= completed <= time.time():
        raise SelectionError("refit chronology is invalid")
    if type(prescribed_epochs) is not int or prescribed_epochs <= 0 or completed_epochs != prescribed_epochs:
        raise SelectionError("refit epoch count is incomplete")
    _verify_refit_manifest(refit_manifest, eligible_ids, int(recipe["epoch_count"]))
    payload = {"recipe_lock": str(recipe_lock),
               "recipe_lock_sha256": _digest(recipe_lock), "weights": str(weights),
               "weights_sha256": _digest(weights), "refit_manifest": str(refit_manifest),
               "refit_manifest_sha256": _digest(refit_manifest),
               "inference_code": [{"path": str(item), "sha256": _digest(item)} for item in inference_code],
               "refit_report": dict(refit_report)}
    return _write_timestamped(path, payload, "created_unix")


def verify_weight_lock(path: Path, weights: Path) -> dict:
    payload = json.loads(path.read_text(encoding="utf-8"))
    if str(weights) != payload["weights"] or _digest(weights) != payload["weights_sha256"]:
        raise SelectionError("weight hash changed after lock")
    for name, hash_name in (("recipe_lock", "recipe_lock_sha256"),
                            ("refit_manifest", "refit_manifest_sha256")):
        target = Path(payload[name])
        if not target.is_file() or _digest(target) != payload[hash_name]:
            raise SelectionError(f"{name} hash changed after lock")
    for item in payload["inference_code"]:
        target = Path(item["path"])
        if not target.is_file() or _digest(target) != item["sha256"]:
            raise SelectionError("inference code hash changed after lock")
    recipe = json.loads(Path(payload["recipe_lock"]).read_text(encoding="utf-8"))
    _verify_recipe_inputs(recipe)
    return payload


def mark_sealed_opened(path: Path, weight_lock: Path, weights: Path) -> str:
    verify_weight_lock(weight_lock, weights)
    return _write_timestamped(path, {"weight_lock_sha256": _digest(weight_lock)},
                              "opened_unix")


def write_sealed_inference_receipt(path: Path, weight_lock: Path, weights: Path,
                                   sealed_opened: Path, baseline_predictions: Path,
                                   candidate_predictions: Path, started_unix: float,
                                   completed_unix: float) -> str:
    locked = verify_weight_lock(weight_lock, weights)
    opened = json.loads(sealed_opened.read_text(encoding="utf-8"))
    if opened.get("weight_lock_sha256") != _digest(weight_lock):
        raise SelectionError("sealed-open receipt does not match weight lock")
    if not locked["created_unix"] <= opened["opened_unix"] <= started_unix <= completed_unix <= time.time():
        raise SelectionError("sealed inference chronology is invalid")
    if not baseline_predictions.is_file() or not candidate_predictions.is_file():
        raise SelectionError("sealed prediction output is missing")
    recipe = json.loads(Path(locked["recipe_lock"]).read_text(encoding="utf-8"))
    payload = {"weight_lock_sha256": _digest(weight_lock), "selected": recipe["selected"],
               "started_unix": started_unix, "completed_unix": completed_unix,
               "baseline_predictions": str(baseline_predictions),
               "baseline_predictions_sha256": _digest(baseline_predictions),
               "candidate_predictions": str(candidate_predictions),
               "candidate_predictions_sha256": _digest(candidate_predictions)}
    return _write_immutable(path, payload)


def _read_predictions(path: Path) -> list[Row]:
    with path.open(encoding="utf-8-sig", newline="") as stream:
        return [dict(row) for row in csv.DictReader(stream)]


def accept_sealed(rows: Sequence[Row], selected: str, weight_lock: Path, weights: Path,
                  sealed_opened: Path, inference_receipt: Path) -> Report:
    if selected not in PIPELINES:
        raise SelectionError("sealed acceptance requires the frozen candidate")
    locked = verify_weight_lock(weight_lock, weights)
    recipe = json.loads(Path(locked["recipe_lock"]).read_text(encoding="utf-8"))
    if selected != recipe["selected"]:
        raise SelectionError("sealed candidate differs from the recipe lock")
    opened = json.loads(sealed_opened.read_text(encoding="utf-8"))
    if opened.get("weight_lock_sha256") != _digest(weight_lock):
        raise SelectionError("sealed-open receipt does not match weight lock")
    receipt = json.loads(inference_receipt.read_text(encoding="utf-8"))
    if receipt.get("weight_lock_sha256") != _digest(weight_lock) or receipt.get("selected") != selected:
        raise SelectionError("sealed inference receipt does not match locked candidate")
    if not locked["created_unix"] <= opened["opened_unix"] <= receipt["started_unix"] <= receipt["completed_unix"] <= time.time():
        raise SelectionError("sealed inference chronology is invalid")
    for name in ("baseline_predictions", "candidate_predictions"):
        target = Path(receipt[name])
        if not target.is_file() or _digest(target) != receipt[f"{name}_sha256"]:
            raise SelectionError("sealed prediction hash changed after inference")
    evaluation = evaluate_sealed(
        rows, _read_predictions(Path(receipt["baseline_predictions"])),
        _read_predictions(Path(receipt["candidate_predictions"])))
    gates = _quality_gates(evaluation)
    accepted = all(gates.values())
    return {"final": selected if accepted else "G0", "candidate": selected,
            "accepted": accepted, "reason": "sealed_passed" if accepted else "sealed_failed",
            "gates": {**gates, "passed": accepted}, "evaluation": evaluation}


In [ ]:
%%writefile /content/dacon_trainable_v4/graph_features.py
"""Shared, file-independent model evidence; baseline preprocessing is retained."""

import gc
import hashlib
import importlib.util
import importlib.metadata
import json
import time
from pathlib import Path

import numpy as np
import torch

from graph_fusion import Feature


def atomic_json(path: Path, value) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    temporary.replace(path)


def digest(path: Path) -> str:
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


class Extractor:
    """Own the three frozen models and accumulate timing for one experiment."""

    def __init__(self, baseline_script: Path, include_raw: bool = True):
        spec = importlib.util.spec_from_file_location('official_baseline', baseline_script)
        if spec is None or spec.loader is None:
            raise ImportError(f'Cannot import baseline at {baseline_script}')
        self.base = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(self.base)
        self.device = self.base.select_device('cuda')
        self.include_raw = include_raw
        torch.manual_seed(20260913)
        np.random.seed(20260913)
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        self.panns, self.vi, self.mi = self.base.load_panns_model(self.device)
        self.df, self.fake_index = self.base.load_df_arena_model(self.device)
        self.demucs = self.base.load_htdemucs_model().to(self.device)
        self.signature = hashlib.sha256((
            digest(Path(__file__)) + digest(baseline_script)
            + (self.base.MODEL_DIR / 'SHA256SUMS.txt').read_text()
            + '|'.join(importlib.metadata.version(k) for k in
                ('torch', 'torchaudio', 'transformers', 'librosa', 'demucs', 'panns-inference'))
            + f'fp32-serial-shifts0-overlap0.25-raw{include_raw}'
        ).encode()).hexdigest()

    def fake_windows(self, audio: np.ndarray) -> list[float]:
        starts = self.base.get_segment_starts(audio.size)
        if self.base.calculate_rms(audio) < self.base.SILENCE_RMS:
            return [0.0] * len(starts)
        result = []
        for start in starts:
            segment = torch.from_numpy(self.base.extract_segment(audio, start)).to(self.device)
            with torch.inference_mode():
                logits = self.df(input_values=segment)['logits']
                probability = torch.softmax(logits.float(), dim=-1)[0, self.fake_index]
            result.append(float(probability))
        return result

    def extract(self, audio_path: Path) -> Feature:
        """No other file or accumulated prediction enters this computation."""
        audio = self.base.load_audio(audio_path)
        predictions, _ = self.panns.inference(self.base.make_panns_segments(audio))
        presence = np.stack((predictions[:, self.vi].max(1), predictions[:, self.mi].max(1)), 1)
        voice, music = self.base.separate_voice_and_music(audio_path, self.demucs, self.device)
        feature = Feature(
            ID=audio_path.stem, starts=self.base.get_segment_starts(audio.size),
            presence=presence.astype(float).tolist(),
            voice_fake=self.fake_windows(voice), music_fake=self.fake_windows(music),
            raw_fake=self.fake_windows(audio) if self.include_raw else [0.0] * len(presence),
            stem_rms=[self.base.calculate_rms(voice), self.base.calculate_rms(music)],
            duration=float(audio.size / self.base.AUDIO_SAMPLE_RATE),
        )
        lengths = {len(feature[name]) for name in ('starts', 'presence', 'voice_fake', 'music_fake', 'raw_fake')}
        if len(lengths) != 1:
            raise ValueError(f'Window alignment mismatch: {audio_path}, {lengths}')
        return feature

    def close(self) -> None:
        del self.panns, self.df, self.demucs
        gc.collect()
        torch.cuda.empty_cache()


def extract_cached(extractor: Extractor, paths: list[Path], cache_dir: Path, deadline: float) -> list[Feature]:
    """Only complete atomic records can resume; content and code changes invalidate keys."""
    features = []
    for path in paths:
        key = hashlib.sha256((digest(path) + extractor.signature).encode()).hexdigest()
        cache_path = cache_dir / f'{key}.json'
        if cache_path.is_file():
            record = json.loads(cache_path.read_text())
            if record['signature'] != extractor.signature or record['input_sha256'] != digest(path):
                raise ValueError(f'Cache provenance mismatch: {cache_path}')
            feature = record['feature']
        else:
            if time.time() >= deadline:
                break
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
            start = time.perf_counter()
            feature = extractor.extract(path)
            torch.cuda.synchronize()
            atomic_json(cache_path, {
                'feature': feature, 'input_sha256': digest(path), 'signature': extractor.signature,
                'seconds': time.perf_counter() - start,
                'max_allocated_gib': torch.cuda.max_memory_allocated() / 1024**3,
                'max_reserved_gib': torch.cuda.max_memory_reserved() / 1024**3,
            })
        features.append({**feature, 'ID': path.stem})
        print(f'FEATURE {len(features)}/{len(paths)} {path.stem}', flush=True)
    return features


In [ ]:
%%writefile /content/dacon_trainable_v4/graph_fusion.py
"""Five file-independent fusion graphs over one shared audio-inference cache."""

from collections.abc import Sequence
from dataclasses import dataclass
from typing import Final, Literal, TypedDict

import numpy as np
from numpy.typing import NDArray


class Feature(TypedDict):
    ID: str
    starts: list[int]
    presence: list[list[float]]
    voice_fake: list[float]
    music_fake: list[float]
    raw_fake: list[float]
    stem_rms: list[float]
    duration: float


class Label(TypedDict):
    ID: str
    group_id: str
    split: Literal["dev", "holdout"]
    FILE_FAKE: int | str | None
    VOICE_FAKE: int | str | None
    MUSIC_FAKE: int | str | None
    VOICE_PRESENT: int | str | None
    MUSIC_PRESENT: int | str | None


class Prediction(TypedDict):
    ID: str
    FILE_FAKE_PROB: float
    VOICE_FAKE_PROB: float
    MUSIC_FAKE_PROB: float
    VOICE_PRESENT_PROB: float
    MUSIC_PRESENT_PROB: float


class Head(TypedDict, total=False):
    status: str
    reason: str
    n: int
    train_ids: list[str]
    coef: list[float]
    intercept: float
    mean: list[float]
    scale: list[float]


class Fusion(TypedDict):
    version: int
    feature_names: list[str]
    train_ids: list[str]
    train_groups: list[str]
    heads: dict[str, Head]


@dataclass(frozen=True, slots=True)
class CacheError(ValueError):
    detail: str

    def __str__(self) -> str:
        return self.detail


FEATURE_NAMES: Final = ["voice_max", "voice_mean", "music_max", "music_mean",
    "raw_max", "raw_mean", "voice_presence_max", "music_presence_max",
    "voice_raw_disagreement", "music_raw_disagreement", "log_windows"]
HEADS: Final = ("FILE_FAKE_PROB", "VOICE_FAKE_PROB", "MUSIC_FAKE_PROB")
FloatArray = NDArray[np.float64]
GraphPredictions = dict[str, Prediction]


def _arrays(feature: Feature) -> tuple[FloatArray, FloatArray]:
    """Reject corrupted cache probabilities and misaligned window evidence."""
    starts = np.asarray(feature["starts"])
    count = len(starts)
    if not isinstance(feature["ID"], str) or not feature["ID"] or starts.ndim != 1 or count == 0:
        raise CacheError("A nonempty ID and nonempty window starts are required")
    if not np.issubdtype(starts.dtype, np.integer) or np.any(starts < 0) or np.any(np.diff(starts) <= 0):
        raise CacheError("Window starts must be strictly increasing nonnegative integers")
    presence = np.asarray(feature["presence"], dtype=np.float64)
    fake = np.asarray([feature[key] for key in ("voice_fake", "music_fake", "raw_fake")], dtype=np.float64)
    if presence.shape != (count, 2) or fake.shape != (3, count):
        raise CacheError("Presence and fake probabilities must align with every window")
    if any(np.any(~np.isfinite(values) | (values < 0) | (values > 1)) for values in (presence, fake)):
        raise CacheError("Cached probabilities must be finite and inside [0, 1]")
    rms = np.asarray(feature["stem_rms"], dtype=np.float64)
    if rms.shape != (2,) or np.any(~np.isfinite(rms) | (rms < 0)):
        raise CacheError("Stem RMS values must be finite and nonnegative")
    if not np.isfinite(feature["duration"]) or feature["duration"] <= 0:
        raise CacheError("Audio duration must be finite and positive")
    return presence, fake


def _vector(presence: FloatArray, fake: FloatArray) -> FloatArray:
    maxima = fake.max(axis=1)
    return np.concatenate((np.column_stack((maxima, fake.mean(axis=1))).ravel(),
        presence.max(axis=0), np.abs(maxima[:2] - maxima[2]), [np.log1p(fake.shape[1])]))


def feature_vector(feature: Feature) -> FloatArray:
    """Return the fixed, file-local 11-feature input used by the learned heads."""
    return _vector(*_arrays(feature))


def _row(identifier: str, values: Sequence[float]) -> Prediction:
    file_fake, voice_fake, music_fake, voice_present, music_present = values
    return {"ID": identifier, "FILE_FAKE_PROB": round(float(file_fake), 10),
        "VOICE_FAKE_PROB": round(float(voice_fake), 10),
        "MUSIC_FAKE_PROB": round(float(music_fake), 10),
        "VOICE_PRESENT_PROB": round(float(voice_present), 10),
        "MUSIC_PRESENT_PROB": round(float(music_present), 10)}


def graph_predictions(feature: Feature, fusion: Fusion | None = None) -> GraphPredictions:
    """Apply all five frozen graphs; G4 falls back per head until fitted."""
    presence, fake = _arrays(feature)
    vp, mp = presence.max(axis=0)
    vf, mf, raw = fake.max(axis=1)
    baseline = _row(feature["ID"], [max(vf * vp, mf * mp), vf, mf, vp, mp])
    top_vf, top_mf = np.sort(fake[:2], axis=1)[:, -min(2, fake.shape[1]):].mean(axis=1)
    aligned = 1.0 - (1.0 - presence[:, 0] * fake[0]) * (1.0 - presence[:, 1] * fake[1])
    hybrid_vf, hybrid_mf = 0.75 * fake[:2].max(axis=1) + 0.25 * raw
    rows = {"G0": baseline,
        "G1": _row(feature["ID"], [max(top_vf * vp, top_mf * mp), top_vf, top_mf, vp, mp]),
        "G2": _row(feature["ID"], [aligned.max(), vf, mf, vp, mp]),
        "G3": _row(feature["ID"], [max(raw, hybrid_vf * vp, hybrid_mf * mp), hybrid_vf, hybrid_mf, vp, mp]),
        "G4": baseline.copy()}
    if fusion is not None:
        if fusion["version"] != 1 or fusion["feature_names"] != FEATURE_NAMES:
            raise CacheError("Fusion model version or feature order is incompatible")
        vector = _vector(presence, fake)
        for name in HEADS:
            head = fusion["heads"][name]
            if head["status"] == "trained":
                coef, mean, scale = (np.asarray(head[key], dtype=np.float64) for key in ("coef", "mean", "scale"))
                if any(a.shape != vector.shape or np.any(~np.isfinite(a)) for a in (coef, mean, scale)) or np.any(scale <= 0) or not np.isfinite(head["intercept"]):
                    raise CacheError("Fusion parameters must be finite with positive scales")
                logit = float(((vector - mean) / scale) @ coef + head["intercept"])
                probability = np.exp(-np.logaddexp(0.0, -logit))
                rows["G4"][name] = round(float(probability), 10)
            elif head["status"] != "fallback":
                raise CacheError("Unknown fusion head status")
    return rows


def _binary(value: int | str | None) -> int | None:
    if value is None or value == "":
        return None
    if value not in (0, 1, "0", "1"):
        raise CacheError("Ground-truth labels must be binary or missing")
    return int(value)


def fit_fusion(features: Sequence[Feature], labels: Sequence[Label]) -> Fusion:
    """Fit C=0.1 logistic heads on eligible dev rows, never held-out sources."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler

    by_id = {f["ID"]: f for f in features}
    if len(by_id) != len(features) or len({row["ID"] for row in labels}) != len(labels):
        raise CacheError("Feature and label IDs must each be unique")
    groups: dict[str, set[str]] = {"dev": set(), "holdout": set()}
    for row in labels:
        if row["split"] not in groups or not isinstance(row["group_id"], str) or not row["group_id"]:
            raise CacheError("Labels require a dev/holdout split and nonempty group_id")
        source_groups = row["group_id"].split("|")
        if not all(source_groups):
            raise CacheError("Mixtures require nonempty constituent source groups")
        groups[row["split"]].update(source_groups)
        for key in ("FILE_FAKE", "VOICE_FAKE", "MUSIC_FAKE", "VOICE_PRESENT", "MUSIC_PRESENT"):
            _binary(row.get(key))
    if groups["dev"] & groups["holdout"]:
        raise CacheError("Source group overlap between dev and holdout")
    development = sorted((row for row in labels if row["split"] == "dev" and row["ID"] in by_id), key=lambda row: row["ID"])
    vectors = {row["ID"]: feature_vector(by_id[row["ID"]]) for row in development}
    result: Fusion = {"version": 1, "feature_names": FEATURE_NAMES.copy(),
        "train_ids": [row["ID"] for row in development],
        "train_groups": sorted({group for row in development for group in row["group_id"].split("|")}), "heads": {}}
    for name, mask in zip(HEADS, (None, "VOICE_PRESENT", "MUSIC_PRESENT")):
        target = name.removesuffix("_PROB")
        eligible = [row for row in development if _binary(row.get(target)) is not None and (mask is None or _binary(row.get(mask)) == 1)]
        y = np.asarray([_binary(row[target]) for row in eligible], dtype=np.int64)
        head: Head = {"status": "fallback", "reason": "fewer_than_two_classes",
            "n": len(eligible), "train_ids": [row["ID"] for row in eligible]}
        if np.unique(y).size == 2:
            x = np.stack([vectors[row["ID"]] for row in eligible])
            scaler = StandardScaler().fit(x)
            model = LogisticRegression(C=0.1, max_iter=1000, random_state=42).fit(scaler.transform(x), y)
            if np.any(model.n_iter_ >= model.max_iter):
                raise CacheError(f"Fusion head {name} failed to converge")
            head.update(status="trained", reason="", coef=model.coef_[0].tolist(),
                intercept=float(model.intercept_[0]), mean=scaler.mean_.tolist(), scale=scaler.scale_.tolist())
        result["heads"][name] = head
    return result


In [ ]:
%%writefile /content/dacon_trainable_v4/graph_scoring.py
"""Submission validation and DACON 236749's discrete ROC scoring rule.

Labels are 0/1/None under the official prediction names without ``_PROB``.
Partial submetrics expose coverage; an official total requires complete labels.
"""

from collections.abc import Mapping, Sequence
from itertools import groupby
from math import isfinite
from typing import Final, TypedDict

PREDICTION_COLUMNS: Final = (
    "FILE_FAKE_PROB", "VOICE_FAKE_PROB", "MUSIC_FAKE_PROB",
    "VOICE_PRESENT_PROB", "MUSIC_PRESENT_PROB",
)
SUBMISSION_COLUMNS: Final = ("ID", *PREDICTION_COLUMNS)
LABEL_COLUMNS: Final = tuple(name.removesuffix("_PROB") for name in PREDICTION_COLUMNS)
Row = Mapping[str, str | float | int | None]


class PredictionRow(TypedDict):
    ID: str
    FILE_FAKE_PROB: float
    VOICE_FAKE_PROB: float
    MUSIC_FAKE_PROB: float
    VOICE_PRESENT_PROB: float
    MUSIC_PRESENT_PROB: float


class MetricsReport(TypedDict):
    score: float | None
    metrics: dict[str, float | None]
    invalid: dict[str, str]
    partial: dict[str, str]
    counts: dict[str, int]
    unknown_counts: dict[str, int]
    coverage: dict[str, float]


def _identifier(row: Row) -> str:
    identifier = row.get("ID")
    if not isinstance(identifier, str) or not identifier or identifier != identifier.strip():
        message = f"ID must be a non-empty, unpadded string: {identifier!r}"
        raise ValueError(message)
    return identifier


def validate_predictions(rows: Sequence[Row], expected_ids: Sequence[str]) -> list[PredictionRow]:
    """Parse probabilities and enforce exact columns, IDs, and submission order."""
    if not rows or len(set(expected_ids)) != len(expected_ids):
        message = "Predictions must be non-empty and expected IDs must be unique"
        raise ValueError(message)
    identifiers = [_identifier(row) for row in rows]
    if len(set(identifiers)) != len(identifiers) or identifiers != list(expected_ids):
        message = "Prediction IDs must match the expected unique IDs in order"
        raise ValueError(message)
    parsed = []
    for row, identifier in zip(rows, identifiers, strict=True):
        if set(row) != set(SUBMISSION_COLUMNS):
            message = f"Prediction {identifier} must have exactly the six submission columns"
            raise ValueError(message)
        probabilities = []
        for column in PREDICTION_COLUMNS:
            try:
                probability = float(row[column])
            except (TypeError, ValueError) as error:
                message = f"Invalid probability for {identifier}/{column}"
                raise ValueError(message) from error
            if not isfinite(probability) or not 0 <= probability <= 1:
                message = f"Probability outside finite [0, 1] for {identifier}/{column}"
                raise ValueError(message)
            probabilities.append(probability)
        parsed.append(PredictionRow(
            ID=identifier, FILE_FAKE_PROB=probabilities[0], VOICE_FAKE_PROB=probabilities[1],
            MUSIC_FAKE_PROB=probabilities[2], VOICE_PRESENT_PROB=probabilities[3],
            MUSIC_PRESENT_PROB=probabilities[4],
        ))
    return parsed


def _label(row: Row, column: str) -> int | None:
    value = row.get(column)
    if value is None or value == "":
        return None
    if value not in (0, 1, "0", "1"):
        message = f"Label must be 0, 1, or unknown for {_identifier(row)}/{column}"
        raise ValueError(message)
    return int(value)


def _roc_metrics(targets: Sequence[int], scores: Sequence[float]) -> tuple[float, float]:
    """Match sklearn's full tied-score ROC and the published nearest-point EER."""
    positives = sum(targets)
    negatives = len(targets) - positives
    points = [(0.0, 0.0)]
    true_positives = false_positives = 0
    ranked = sorted(zip(scores, targets, strict=True), reverse=True)
    for _, tied in groupby(ranked, key=lambda pair: pair[0]):
        for _, target in tied:
            true_positives += target
            false_positives += 1 - target
        points.append((false_positives / negatives, true_positives / positives))
    fpr, tpr = min(points, key=lambda point: abs(point[0] - (1 - point[1])))
    fnr = 1 - tpr
    eer = (fpr + fnr) / 2
    auc = sum(
        (right[0] - left[0]) * (right[1] + left[1]) / 2
        for left, right in zip(points, points[1:])
    )
    return eer, auc


def official_metrics(labels: Sequence[Row], predictions: Sequence[Row]) -> MetricsReport:
    """Report known-subset metrics, but require complete truth for an official total.

    Coverage is known eligible labels / (known eligible + unknown relevant labels).
    Unknown presence is relevant because it leaves component eligibility unknown.
    """
    parsed = validate_predictions(predictions, [_identifier(row) for row in labels])
    truth = [{column: _label(row, column) for column in LABEL_COLUMNS} for row in labels]
    report = MetricsReport(
        score=None, metrics={}, invalid={}, partial={}, counts={}, unknown_counts={}, coverage={},
    )
    specifications = (
        ("file_eer", "FILE_FAKE", "FILE_FAKE_PROB", None),
        ("voice_eer", "VOICE_FAKE", "VOICE_FAKE_PROB", "VOICE_PRESENT"),
        ("music_eer", "MUSIC_FAKE", "MUSIC_FAKE_PROB", "MUSIC_PRESENT"),
        ("voice_presence_auc", "VOICE_PRESENT", "VOICE_PRESENT_PROB", None),
        ("music_presence_auc", "MUSIC_PRESENT", "MUSIC_PRESENT_PROB", None),
    )
    for name, target_column, score_column, mask_column in specifications:
        targets = []
        scores = []
        missing = 0
        for row, prediction in zip(truth, parsed, strict=True):
            if mask_column is not None:
                if row[mask_column] is None:
                    missing += 1
                    continue
                if row[mask_column] == 0:
                    continue
            target = row[target_column]
            if target is None:
                missing += 1
                continue
            targets.append(target)
            scores.append(float(prediction[score_column]))
        report["counts"][name] = len(targets)
        report["unknown_counts"][name] = missing
        denominator = len(targets) + missing
        report["coverage"][name] = len(targets) / denominator if denominator else 0.0
        report["metrics"][name] = None
        if missing:
            report["partial"][name] = f"known subset only; unknown relevant labels: {missing}"
        if len(set(targets)) != 2:
            report["invalid"][name] = "both positive and negative labels are required"
            continue
        eer, auc = _roc_metrics(targets, scores)
        report["metrics"][name] = auc if name.endswith("auc") else eer
    if not report["invalid"] and not report["partial"]:
        values = [value for value in report["metrics"].values() if value is not None]
        file_eer, voice_eer, music_eer, voice_auc, music_auc = values
        report["score"] = (
            0.45 * (1 - file_eer) + 0.18 * (1 - voice_eer) + 0.27 * (1 - music_eer)
            + 0.05 * voice_auc + 0.05 * music_auc
        )
    return report


In [ ]:
%%writefile /content/dacon_trainable_v4/v2_scoring.py
"""Exact ROC diagnostics and development-only model selection for V2."""

from __future__ import annotations

import argparse, csv, json, random
from collections import defaultdict
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Final

from graph_scoring import official_metrics, validate_predictions

Scalar = str | int | float | None
Row = Mapping[str, Scalar]
PredictionRows = Mapping[str, Sequence[Row]]
WEIGHTS: Final = {"file_eer": .45, "voice_eer": .18, "music_eer": .27, "voice_presence_auc": .05, "music_presence_auc": .05}
EER_SPECS: Final = (("file", "FILE_FAKE", "FILE_FAKE_PROB", None), ("voice", "VOICE_FAKE", "VOICE_FAKE_PROB", "VOICE_PRESENT"), ("music", "MUSIC_FAKE", "MUSIC_FAKE_PROB", "MUSIC_PRESENT"))


def _label(row: Row, name: str) -> int | None:
    value = row.get(name)
    if value is None or value == "":
        return None
    if value not in (0, 1, "0", "1"):
        raise ValueError(f"invalid binary label: {name}")
    return int(value)


def _role(row: Row) -> str:
    role = row.get("role", row.get("split"))
    if not isinstance(role, str) or not role:
        raise ValueError("manifest rows require role or split")
    return role


def _prediction_map(rows: Sequence[Row], manifest: Sequence[Row]) -> dict[str, Row]:
    expected = [str(row["ID"]) for row in manifest]
    parsed = validate_predictions(rows, expected)
    return {row["ID"]: row for row in parsed}


def _eligible(rows: Sequence[Row], predictions: Mapping[str, Row], target: str,
              score: str, mask: str | None) -> tuple[list[int], list[float], int]:
    targets: list[int] = []
    scores: list[float] = []
    unknown = 0
    for row in rows:
        if mask is not None:
            present = _label(row, mask)
            if present is None:
                unknown += 1
                continue
            if present == 0:
                continue
        truth = _label(row, target)
        if truth is None:
            unknown += 1
            continue
        targets.append(truth)
        scores.append(float(predictions[str(row["ID"])][score]))
    return targets, scores, unknown


def _roc(targets: Sequence[int], scores: Sequence[float]) -> dict[str, Scalar] | None:
    positives = sum(targets)
    negatives = len(targets) - positives
    if not positives or not negatives:
        return None
    ranked = sorted(zip(scores, targets, strict=True), reverse=True)
    points: list[tuple[float | None, int, int]] = [(None, 0, 0)]
    tp = fp = 0
    index = 0
    while index < len(ranked):
        threshold = ranked[index][0]
        while index < len(ranked) and ranked[index][0] == threshold:
            tp += ranked[index][1]
            fp += 1 - ranked[index][1]
            index += 1
        points.append((threshold, tp, fp))
    def distance(point: tuple[float | None, int, int]) -> float:
        return abs(point[2] / negatives - (1 - point[1] / positives))
    threshold, tp, fp = min(points, key=distance)
    fpr, fnr = fp / negatives, 1 - tp / positives
    auc = sum((right[2] / negatives - left[2] / negatives) *
              (right[1] / positives + left[1] / positives) / 2
              for left, right in zip(points, points[1:]))
    return {"eer": (fpr + fnr) / 2, "auc": auc, "threshold": threshold,
            "fpr": fpr, "fnr": fnr, "fp": fp, "fn": positives - tp,
            "tp": tp, "tn": negatives - fp, "n": len(targets)}


def _detail_report(rows: Sequence[Row], predictions: Mapping[str, Row]) -> dict[str, dict[str, Scalar]]:
    details: dict[str, dict[str, Scalar]] = {}
    for name, target, score, mask in EER_SPECS:
        targets, scores, unknown = _eligible(rows, predictions, target, score, mask)
        point = _roc(targets, scores)
        details[name] = {"unknown_relevant": unknown, "n": len(targets),
                         "point": point}
    return details


def _pair_changes(rows: Sequence[Row], baseline: Mapping[str, Row], candidate: Mapping[str, Row],
                  target: str, score: str, mask: str | None) -> dict[str, int]:
    labels, scores_a, _ = _eligible(rows, baseline, target, score, mask)
    _, scores_b, _ = _eligible(rows, candidate, target, score, mask)
    positives = [index for index, value in enumerate(labels) if value == 1]
    negatives = [index for index, value in enumerate(labels) if value == 0]
    flips = worsened = 0
    for positive in positives:
        for negative in negatives:
            before = scores_a[positive] - scores_a[negative]
            after = scores_b[positive] - scores_b[negative]
            if (before > 0 > after) or (before < 0 < after):
                flips += 1
                worsened += int(before > 0 > after)
    return {"pairs": len(positives) * len(negatives), "flips": flips,
            "positive_to_negative_worsened": worsened}


def audit_forensics(manifest_rows: Sequence[Row], predictions_by_name: PredictionRows) -> dict[str, object]:
    """Report all available roles without converting a partial proxy into a score."""
    maps = {name: _prediction_map(rows, manifest_rows) for name, rows in predictions_by_name.items()}
    if "G0" not in maps:
        raise ValueError("G0 predictions are required for forensics")
    roles: dict[str, list[Row]] = defaultdict(list)
    for row in manifest_rows:
        roles[_role(row)].append(row)
    report: dict[str, object] = {"metric": "nearest_abs_fpr_minus_fnr", "roles": {}}
    for role, rows in sorted(roles.items()):
        models: dict[str, object] = {}
        for name, prediction in maps.items():
            models[name] = {"official_proxy_only": official_metrics(rows, [prediction[str(row["ID"])] for row in rows]),
                            "roc": _detail_report(rows, prediction)}
        comparisons: dict[str, object] = {}
        for name, prediction in maps.items():
            if name == "G0":
                continue
            per_head: dict[str, object] = {}
            for head, target, score, mask in EER_SPECS:
                base = models["G0"]["roc"][head]["point"]
                changed = models[name]["roc"][head]["point"]
                per_head[head] = {"baseline": base, "candidate": changed,
                                  "auc_delta": None if base is None or changed is None else changed["auc"] - base["auc"],
                                  "pair_changes": _pair_changes(rows, maps["G0"], prediction, target, score, mask)}
            comparisons[name] = per_head
        report["roles"][role] = {"models": models, "comparisons": comparisons}
    return report


def _proxy(rows: Sequence[Row], predictions: Mapping[str, Row]) -> dict[str, object]:
    metrics = official_metrics(rows, [predictions[str(row["ID"])] for row in rows])
    usable = {name: weight for name, weight in WEIGHTS.items() if metrics["metrics"][name] is not None}
    total = sum(usable.values())
    values = metrics["metrics"]
    proxy = None if not total else sum(weight / total * ((1 - float(values[name])) if name.endswith("eer") else float(values[name]))
                                      for name, weight in usable.items())
    return {"value": proxy, "proxy_weights": usable, "metric_report": metrics,
            "is_partial": bool(metrics["partial"])}


def _resampled_proxy(rows: Sequence[Row], predictions: Mapping[str, Row]) -> float | None:
    """Score a bootstrap sample whose repeated IDs cannot enter CSV validation."""
    values: dict[str, float] = {}
    for metric, target, score, mask in EER_SPECS:
        targets, scores, _ = _eligible(rows, predictions, target, score, mask)
        point = _roc(targets, scores)
        if point is not None:
            values[f"{metric}_eer"] = point["eer"]
    for metric, target, score in (("voice_presence", "VOICE_PRESENT", "VOICE_PRESENT_PROB"),
                                  ("music_presence", "MUSIC_PRESENT", "MUSIC_PRESENT_PROB")):
        targets, scores, _ = _eligible(rows, predictions, target, score, None)
        point = _roc(targets, scores)
        if point is not None:
            values[f"{metric}_auc"] = point["auc"]
    usable = {name: weight for name, weight in WEIGHTS.items() if name in values}
    total = sum(usable.values())
    return None if not total else sum(weight / total * ((1 - values[name]) if name.endswith("eer") else values[name])
                                     for name, weight in usable.items())


def _components(rows: Sequence[Row]) -> list[list[int]]:
    parent: dict[str, str] = {}
    def find(value: str) -> str:
        parent.setdefault(value, value)
        while parent[value] != value:
            parent[value] = parent[parent[value]]
            value = parent[value]
        return value
    def join(left: str, right: str) -> None:
        root_left, root_right = find(left), find(right)
        if root_left != root_right:
            parent[root_right] = root_left
    for row in rows:
        raw_group = row.get("group_id")
        if not isinstance(raw_group, str) or not raw_group:
            raise ValueError("manifest rows require nonempty group_id constituents")
        groups = raw_group.split("|")
        if any(not group for group in groups):
            raise ValueError("manifest rows require nonempty group_id constituents")
        for group in groups[1:]:
            join(groups[0], group)
    units: dict[str, list[int]] = defaultdict(list)
    for index, row in enumerate(rows):
        units[find(str(row["group_id"]).split("|")[0])].append(index)
    return list(units.values())


def _bootstrap(rows: Sequence[Row], baseline: Mapping[str, Row], candidate: Mapping[str, Row]) -> dict[str, object]:
    units = _components(rows)
    rng = random.Random(20260913)
    deltas: list[float] = []
    for _ in range(2000):
        sampled = [row for unit in (rng.choice(units) for _ in units) for index in unit for row in [rows[index]]]
        base, changed = _resampled_proxy(sampled, baseline), _resampled_proxy(sampled, candidate)
        if base is not None and changed is not None:
            deltas.append(changed - base)
    deltas.sort()
    return {"seed": 20260913, "resamples": 2000, "lineage_components": len(units),
            "valid_resamples": len(deltas), "uncomputable_resamples": 2000 - len(deltas),
            "win_rate": None if not deltas else sum(delta > 0 for delta in deltas) / len(deltas),
            "p10_delta": None if not deltas else deltas[max(0, (len(deltas) - 1) // 10)]}


def select_candidate(manifest_rows: Sequence[Row], predictions_by_name: PredictionRows) -> dict[str, object]:
    """Select a V2 candidate only from the dedicated development selector role."""
    if any(_role(row) != "dev_select" for row in manifest_rows):
        raise ValueError("selection accepts development roles only")
    maps = {name: _prediction_map(rows, manifest_rows) for name, rows in predictions_by_name.items()}
    if "G0" not in maps:
        raise ValueError("G0 predictions are required for selection")
    models: dict[str, dict[str, object]] = {"G0": _proxy(manifest_rows, maps["G0"])}
    candidates = [name for name in ("AST_FROZEN", "AST_LAST_BLOCK", "G0_AST_MUSIC") if name in maps]
    for name in candidates:
        current = _proxy(manifest_rows, maps[name])
        bootstrap = _bootstrap(manifest_rows, maps["G0"], maps[name])
        current["bootstrap"] = bootstrap
        base_metrics = models["G0"]["metric_report"]["metrics"]
        new_metrics = current["metric_report"]["metrics"]
        regression = any(float(new_metrics[key]) > float(base_metrics[key]) + .02 for key in WEIGHTS if key.endswith("eer") and new_metrics[key] is not None and base_metrics[key] is not None)
        companion = any(_roc(*_eligible(manifest_rows, maps[name], target, score, mask)[:2])["auc"] < _roc(*_eligible(manifest_rows, maps["G0"], target, score, mask)[:2])["auc"] - .01 for _, target, score, mask in EER_SPECS if _roc(*_eligible(manifest_rows, maps[name], target, score, mask)[:2]) is not None and _roc(*_eligible(manifest_rows, maps["G0"], target, score, mask)[:2]) is not None)
        gain = None if current["value"] is None or models["G0"]["value"] is None else float(current["value"]) - float(models["G0"]["value"])
        gates = {"proxy_gain": gain is not None and gain >= .005,
                 "bootstrap_complete": bootstrap["valid_resamples"] == 2000,
                 "bootstrap_win_rate": bootstrap["win_rate"] is not None and bootstrap["win_rate"] >= .70,
                 "bootstrap_p10": bootstrap["p10_delta"] is not None and bootstrap["p10_delta"] >= -.005,
                 "eer_regression": not regression, "companion_auc_regression": not companion}
        current["gates"] = {**gates, "proxy_gain_delta": gain, "passed": all(gates.values())}
        models[name] = current
    passed = [name for name in candidates if models[name]["gates"]["passed"]]
    selected = max(passed, key=lambda name: (models[name]["value"], -candidates.index(name)), default="G0")
    failed = {name: [gate for gate, value in models[name]["gates"].items() if value is False]
              for name in candidates if name not in passed}
    return {"selected": selected, "reason": "all_gates_passed" if selected != "G0" else "no_candidate_passed_all_gates",
            "failed_gates": failed, "models": models}


def _read_csv(path: Path) -> list[dict[str, Scalar]]:
    with path.open(encoding="utf-8-sig", newline="") as stream:
        return [dict(row) for row in csv.DictReader(stream)]


def _load_predictions(directory: Path, manifest: Sequence[Row]) -> dict[str, list[Row]]:
    files = {name: directory / f"{name}_predictions.csv" for name in ("G0", "G1", "G2", "G3", "G4")}
    if all(path.is_file() for path in files.values()):
        return {name: _read_csv(path) for name, path in files.items()}
    from graph_fusion import graph_predictions
    features = json.loads((directory / "features.json").read_text())
    fusion = json.loads((directory / "fusion.json").read_text())
    by_id = {feature["ID"]: graph_predictions(feature, fusion) for feature in features}
    return {name: [by_id[str(row["ID"])][name] for row in manifest] for name in files}


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("diagnose")
    parser.add_argument("--manifest", type=Path, required=True)
    parser.add_argument("--predictions", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    args = parser.parse_args()
    manifest = _read_csv(args.manifest)
    report = audit_forensics(manifest, _load_predictions(args.predictions, manifest))
    args.output.parent.mkdir(parents=True, exist_ok=True)
    args.output.write_text(json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False) + "\n")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/dacon_trainable_v4/v2_data.py
"""Small shared data/audio primitives for the V2 bank."""
from __future__ import annotations

import hashlib
import bz2
import subprocess
import struct
import wave
import zlib
from collections.abc import Callable, Sequence
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Final, TypeVar, assert_never

import imageio_ffmpeg
import numpy as np

LABEL_NAMES: Final = ("FILE_FAKE", "VOICE_FAKE", "MUSIC_FAKE", "VOICE_PRESENT", "MUSIC_PRESENT")
Candidate = TypeVar("Candidate")
Loaded = TypeVar("Loaded")


class DataIntegrityError(RuntimeError):
    """A retained data item failed an integrity boundary."""


@dataclass(frozen=True, slots=True)
class Labels:
    FILE_FAKE: str
    VOICE_FAKE: str
    MUSIC_FAKE: str
    VOICE_PRESENT: str
    MUSIC_PRESENT: str


@dataclass(frozen=True, slots=True)
class SourceRow:
    id: str
    prepared_path: str
    original_path: str
    source_kind: str
    group_id: str
    provider: str
    content_id: str
    source_url: str
    revision: str
    source_sha256: str
    pcm_sha256: str
    license: str
    attribution: str
    labels: Labels

    def json_record(self) -> dict[str, str | dict[str, str]]:
        return asdict(self)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def decode_audio(path: Path, limit_seconds: int | None = 10) -> np.ndarray:
    result = subprocess.run(
        [imageio_ffmpeg.get_ffmpeg_exe(), "-v", "error", "-i", str(path), "-f", "f32le", "-ac", "1", "-ar", "16000", "-"],
        check=True, capture_output=True,
    )
    audio = np.frombuffer(result.stdout, dtype="<f4").copy()
    if not np.isfinite(audio).all():
        raise ArithmeticError(f"non-finite decoded audio: {path}")
    if limit_seconds is None:
        return audio
    limit = limit_seconds * 16000
    start = max(0, (len(audio) - limit) // 2)
    return audio[start:start + limit]


def pcm_sha256(audio: np.ndarray) -> str:
    normalized = np.asarray(audio, dtype="<f4")
    return hashlib.sha256(normalized.tobytes()).hexdigest()


def write_wav(path: Path, audio: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    peak = float(np.max(np.abs(audio))) if len(audio) else 0.0
    pcm = np.round(audio / max(1.0, peak / 0.98) * 32767).astype("<i2")
    with wave.open(str(path), "wb") as handle:
        handle.setparams((1, 2, 16000, 0, "NONE", "not compressed"))
        handle.writeframes(pcm.tobytes())


def stable_key(value: str) -> str:
    return hashlib.sha256(value.encode()).hexdigest()


def first_valid(candidates: Sequence[Candidate], loader: Callable[[Candidate], Loaded | None], count: int) -> list[Loaded]:
    accepted: list[Loaded] = []
    for start in range(0, len(candidates), 8):
        with ThreadPoolExecutor(max_workers=8) as executor:
            accepted.extend(item for item in executor.map(loader, candidates[start:start + 8]) if item is not None)
        if len(accepted) >= count:
            return accepted[:count]
    return accepted


def extract_local_member(blob: bytes, entry: dict[str, int | str], output: Path) -> Path:
    if len(blob) < 30:
        raise DataIntegrityError("ZIP member local header is truncated")
    signature, _, _, method, _, _, _, _, _, name_len, extra_len = struct.unpack("<4s5H3L2H", blob[:30])
    if signature != b"PK\x03\x04" or method != int(entry["compress_type"]):
        raise DataIntegrityError("ZIP member header or compression method mismatch")
    start = 30 + name_len + extra_len
    compressed = blob[start:start + int(entry["compress_size"])]
    if method not in (0, 8, 12):
        raise DataIntegrityError(f"unsupported ZIP compression method {method}")
    match method:
        case 0:
            payload = compressed
        case 8:
            payload = zlib.decompress(compressed, -15)
        case 12:
            payload = bz2.decompress(compressed)
        case unreachable:
            assert_never(unreachable)
    if len(payload) != int(entry["file_size"]):
        raise DataIntegrityError("ZIP member size mismatch")
    if zlib.crc32(payload) != int(entry["CRC"]):
        raise DataIntegrityError("ZIP member CRC mismatch")
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_bytes(payload)
    return output


In [ ]:
%%writefile /content/dacon_trainable_v4/recovery_sonics.py
"""Pinned, file-local SONICS music-authenticity inference."""

from __future__ import annotations

import hashlib
import math
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Final, TypedDict

import numpy as np
import torch

SAMPLE_RATE: Final = 16_000
WINDOW_SAMPLES: Final = SAMPLE_RATE * 5
CONFIG_SHA256: Final = "754bc15b3f914afaf4e002e50bfaa5e81055c9fb0e2b76408e52fd88f09f7def"
WEIGHTS_SHA256: Final = "11c384cea651c56028d6f02e04aa467c2099cf1102687353fe4dd9504c4db868"


class SonicsPrediction(TypedDict):
    mean_probability: float
    window_probabilities: list[float]
    starts: list[int]
    covered_samples: int
    seconds: float
    audio_seconds: float
    elapsed_seconds: float


@dataclass(frozen=True, slots=True)
class SonicsContractError(ValueError):
    detail: str

    def __str__(self) -> str:
        return self.detail


def _sha256(path: Path) -> str:
    """Return one asset's full SHA-256 digest."""
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def _verified_assets(assets: Path) -> None:
    """Reject a missing or altered local asset before checkpoint deserialization."""
    for name, expected in (("config.json", CONFIG_SHA256), ("pytorch_model.bin", WEIGHTS_SHA256)):
        path = assets / name
        if not path.is_file():
            raise SonicsContractError(f"SONICS local asset is missing: {path}")
        if _sha256(path) != expected:
            raise SonicsContractError(f"SONICS {name} SHA-256 mismatch")


class SonicsPredictor:
    """Run the pinned classifier serially over one 16 kHz mono waveform."""

    model: torch.nn.Module
    device: torch.device

    def __init__(self, assets: Path, device: str) -> None:
        _verified_assets(assets)
        vendor_root = assets.parent / "sonics_vendor"
        if not (vendor_root / "sonics").is_dir():
            raise SonicsContractError(f"Pinned SONICS source is missing beside assets: {vendor_root}")
        sys.path.insert(0, str(vendor_root))
        from sonics.models.hf_model import HFAudioClassifier

        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        self.device = torch.device(device)
        self.model = HFAudioClassifier.from_pretrained(str(assets), map_location="cpu", strict=True).to(self.device).eval()

    def predict(self, audio: np.ndarray) -> SonicsPrediction:
        """Return a mean of serial 5-second window probabilities for one file."""
        waveform = np.asarray(audio)
        if waveform.ndim != 1 or waveform.size == 0:
            raise SonicsContractError("SONICS audio must be nonempty and one-dimensional")
        try:
            waveform = np.ascontiguousarray(waveform, dtype=np.float32)
        except (TypeError, ValueError) as error:
            raise SonicsContractError("SONICS audio must be convertible to float32") from error
        if not np.isfinite(waveform).all():
            raise SonicsContractError("SONICS audio must contain only finite values")
        starts = self._window_starts(waveform.size)
        started = time.perf_counter()
        probabilities = [self._window_probability(waveform[start : start + WINDOW_SAMPLES]) for start in starts]
        elapsed = time.perf_counter() - started
        if not all(math.isfinite(value) and 0.0 <= value <= 1.0 for value in probabilities):
            raise SonicsContractError("SONICS model returned a non-finite probability")
        covered_samples = self._covered_samples(starts, waveform.size)
        if covered_samples != waveform.size:
            raise SonicsContractError("SONICS windows must cover every original sample")
        audio_seconds = waveform.size / SAMPLE_RATE
        return {
            "mean_probability": float(np.mean(probabilities)),
            "window_probabilities": probabilities,
            "starts": starts,
            "covered_samples": covered_samples,
            "seconds": audio_seconds,
            "audio_seconds": audio_seconds,
            "elapsed_seconds": elapsed,
        }

    @staticmethod
    def _window_starts(samples: int) -> list[int]:
        """Cover the tail with a final full window; only short files are padded."""
        if samples <= WINDOW_SAMPLES:
            return [0]
        final_start = samples - WINDOW_SAMPLES
        starts = list(range(0, final_start + 1, WINDOW_SAMPLES))
        if starts[-1] != final_start:
            starts.append(final_start)
        return starts

    @staticmethod
    def _covered_samples(starts: list[int], samples: int) -> int:
        """Measure the covered union, rejecting a gap rather than reporting input length."""
        covered_end = 0
        for start in starts:
            if start > covered_end:
                return covered_end
            covered_end = max(covered_end, min(samples, start + WINDOW_SAMPLES))
        return covered_end

    def _window_probability(self, samples: np.ndarray) -> float:
        """Infer exactly one window, preserving per-example feature normalization."""
        padded = np.zeros(WINDOW_SAMPLES, dtype=np.float32)
        padded[: samples.size] = samples
        with torch.no_grad():
            logits = self.model(torch.from_numpy(padded).unsqueeze(0).to(self.device))
        if logits.numel() != 1:
            raise SonicsContractError("SONICS model must return exactly one logit per window")
        logit = float(logits.item())
        if not math.isfinite(logit):
            raise SonicsContractError("SONICS model returned a non-finite logit")
        return float(torch.sigmoid(torch.tensor(logit)).item())


In [ ]:
%%writefile /content/dacon_trainable_v4/ast_multitask.py
"""Pinned AST encoder with five independently trainable DACON heads."""

from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Final, Literal, assert_never

import numpy as np
import torch
import torch.nn.functional as functional
from safetensors.torch import load_file, save_file
from torch import nn
from transformers import ASTConfig, ASTFeatureExtractor, ASTModel

PREDICTION_COLUMNS: Final = (
    "FILE_FAKE_PROB",
    "VOICE_FAKE_PROB",
    "MUSIC_FAKE_PROB",
    "VOICE_PRESENT_PROB",
    "MUSIC_PRESENT_PROB",
)
SAMPLE_RATE: Final = 16_000
WINDOW_SAMPLES: Final = 160_000
PROCESSOR_CONTRACT: Final = {
    "sampling_rate": SAMPLE_RATE,
    "num_mel_bins": 128,
    "max_length": 1024,
    "mean": -4.2677393,
    "std": 4.5689974,
}
TrainMode = Literal["frozen", "last_block"]


class ASTContractError(ValueError):
    """An offline AST asset or input violates the pinned model contract."""


def _validate_processor(processor: ASTFeatureExtractor) -> None:
    for name, expected in PROCESSOR_CONTRACT.items():
        actual = getattr(processor, name, None)
        if not isinstance(actual, int | float) or not math.isclose(float(actual), expected):
            raise ASTContractError("AST processor contract mismatch")


def load_processor(local_dir: Path) -> ASTFeatureExtractor:
    """Load the pinned processor locally and fail closed on configuration drift."""
    processor = ASTFeatureExtractor.from_pretrained(local_dir, local_files_only=True)
    _validate_processor(processor)
    return processor


def _mps_block_input_gradient(module: nn.Module, inputs: tuple) -> tuple:
    """Avoid MPS partial LayerNorm backward errors without unfreezing the prefix."""
    hidden = inputs[0]
    if (hidden.device.type == "mps" and torch.is_grad_enabled()
            and module.layernorm_before.weight.requires_grad):
        hidden.requires_grad_(True)
    return inputs


class ASTMultiTask(nn.Module):
    """AST pooler followed by five independent binary heads."""

    def __init__(self, backbone: ASTModel, mode: TrainMode) -> None:
        super().__init__()
        self.backbone = backbone
        self.backbone.encoder.layer[-1].register_forward_pre_hook(_mps_block_input_gradient)
        self.heads = nn.ModuleList(nn.Linear(backbone.config.hidden_size, 1) for _ in PREDICTION_COLUMNS)
        self.mode = mode
        self.configure_trainable(mode)

    @classmethod
    def from_pretrained(cls, local_dir: Path, mode: TrainMode) -> ASTMultiTask:
        """Load only supplied local AST weights; AudioSet classifier logits are unused."""
        load_processor(local_dir)
        backbone = ASTModel.from_pretrained(
            local_dir,
            local_files_only=True,
            attn_implementation="sdpa",
        )
        return cls(backbone, mode)

    def configure_trainable(self, mode: TrainMode) -> None:
        """Expose exactly the parameters owned by the selected training mode."""
        for parameter in self.parameters():
            parameter.requires_grad_(False)
        for parameter in self.heads.parameters():
            parameter.requires_grad_(True)
        match mode:
            case "frozen":
                pass
            case "last_block":
                for parameter in self.backbone.encoder.layer[-1].parameters():
                    parameter.requires_grad_(True)
                for parameter in self.backbone.layernorm.parameters():
                    parameter.requires_grad_(True)
            case unreachable:
                assert_never(unreachable)
        self.mode = mode

    def train(self, mode: bool = True) -> ASTMultiTask:
        """Keep frozen encoder blocks deterministic while training selected layers."""
        super().train(mode)
        if mode:
            self.backbone.eval()
            if self.mode == "last_block":
                self.backbone.encoder.layer[-1].train()
        return self

    def encode(self, input_values: torch.Tensor) -> torch.Tensor:
        """Return the final-normalized CLS/distillation-token pooler output."""
        return self.backbone(input_values=input_values).pooler_output

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        pooled = self.encode(input_values)
        return torch.cat([head(pooled) for head in self.heads], dim=-1)


def _eligible_mask(targets: torch.Tensor) -> torch.Tensor:
    if targets.ndim != 2 or targets.shape[1] != len(PREDICTION_COLUMNS):
        raise ASTContractError("AST targets must have five columns")
    known = torch.isfinite(targets)
    if torch.any(known & (targets != 0) & (targets != 1)):
        raise ASTContractError("AST targets must be binary or NaN")
    eligible = known.clone()
    eligible[:, 1] &= known[:, 3] & (targets[:, 3] == 1)
    eligible[:, 2] &= known[:, 4] & (targets[:, 4] == 1)
    return eligible


def compute_pos_weights(targets: torch.Tensor) -> torch.Tensor:
    """Compute bounded training-only positive weights over eligible truth."""
    eligible = _eligible_mask(targets)
    weights = torch.ones(len(PREDICTION_COLUMNS), dtype=targets.dtype, device=targets.device)
    for index in range(len(PREDICTION_COLUMNS)):
        selected = targets[:, index][eligible[:, index]]
        positives = torch.count_nonzero(selected == 1)
        negatives = torch.count_nonzero(selected == 0)
        if positives > 0 and negatives > 0:
            weights[index] = (negatives / positives).clamp(0.25, 4.0)
    return weights


def masked_multitask_loss(
    logits: torch.Tensor,
    targets: torch.Tensor,
    pos_weights: torch.Tensor,
) -> torch.Tensor:
    """Mean five independently normalized BCE heads over known eligible labels."""
    if logits.shape != targets.shape or pos_weights.shape != (len(PREDICTION_COLUMNS),):
        raise ASTContractError("AST loss input contract mismatch")
    eligible = _eligible_mask(targets)
    losses: list[torch.Tensor] = []
    safe_targets = torch.nan_to_num(targets)
    for index in range(len(PREDICTION_COLUMNS)):
        mask = eligible[:, index]
        if torch.any(mask):
            losses.append(functional.binary_cross_entropy_with_logits(
                logits[:, index][mask],
                safe_targets[:, index][mask],
                pos_weight=pos_weights[index],
            ))
    if not losses:
        raise ASTContractError("AST batch has no eligible labels")
    return torch.stack(losses).mean()


def split_audio(
    audio: np.ndarray,
    sample_rate: int = SAMPLE_RATE,
    window_samples: int = WINDOW_SAMPLES,
) -> tuple[np.ndarray, ...]:
    """Cover one mono file with deterministic, non-overlapping windows including its tail."""
    if sample_rate != SAMPLE_RATE or window_samples <= 0 or audio.ndim != 1 or audio.size == 0:
        raise ASTContractError("AST audio input contract mismatch")
    if not np.isfinite(audio).all():
        raise ASTContractError("AST audio contains non-finite samples")
    return tuple(audio[start:start + window_samples] for start in range(0, audio.size, window_samples))


def prepare_inputs(
    processor: ASTFeatureExtractor,
    windows: tuple[np.ndarray, ...],
) -> torch.Tensor:
    """Apply the pinned AST feature extractor to one file's windows."""
    _validate_processor(processor)
    if not windows:
        raise ASTContractError("AST requires at least one audio window")
    batch = processor(list(windows), sampling_rate=SAMPLE_RATE, return_tensors="pt")
    return batch["input_values"]


def aggregate_window_logits(logits: torch.Tensor) -> torch.Tensor:
    """Mean all window probabilities from exactly one file."""
    if logits.ndim != 2 or logits.shape[0] == 0 or logits.shape[1] != len(PREDICTION_COLUMNS):
        raise ASTContractError("AST window logits contract mismatch")
    return torch.sigmoid(logits.float()).mean(dim=0)


def predict_file(
    model: ASTMultiTask,
    processor: ASTFeatureExtractor,
    audio: np.ndarray,
) -> torch.Tensor:
    """Predict one file without reading or retaining another file's state."""
    inputs = prepare_inputs(processor, split_audio(audio)).to(next(model.parameters()).device)
    with torch.inference_mode():
        return aggregate_window_logits(model(inputs)).cpu()


def export_checkpoint(
    model: ASTMultiTask,
    processor: ASTFeatureExtractor,
    directory: Path,
) -> Path:
    """Export a self-contained inference checkpoint without optimizer state."""
    _validate_processor(processor)
    directory.mkdir(parents=True, exist_ok=True)
    model.backbone.config.to_json_file(directory / "config.json")
    processor.save_pretrained(directory)
    metadata = {
        "format_version": 1,
        "mode": model.mode,
        "columns": list(PREDICTION_COLUMNS),
        "hidden_size": model.backbone.config.hidden_size,
        "processor_contract": PROCESSOR_CONTRACT,
    }
    metadata_path = directory / "multitask_config.json"
    temporary_metadata = metadata_path.with_suffix(".json.tmp")
    temporary_metadata.write_text(json.dumps(metadata, sort_keys=True), encoding="utf-8")
    temporary_metadata.replace(metadata_path)
    weights_path = directory / "model.safetensors"
    temporary_weights = weights_path.with_suffix(".safetensors.tmp")
    save_file(
        {name: value.detach().cpu().contiguous() for name, value in model.state_dict().items()},
        temporary_weights,
    )
    temporary_weights.replace(weights_path)
    return directory


def load_checkpoint(
    directory: Path,
    mode: TrainMode,
) -> tuple[ASTMultiTask, ASTFeatureExtractor]:
    """Load a self-contained checkpoint and reject processor/head contract drift."""
    metadata_path = directory / "multitask_config.json"
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    expected = {
        "format_version": 1,
        "mode": mode,
        "columns": list(PREDICTION_COLUMNS),
    }
    if any(metadata.get(name) != value for name, value in expected.items()):
        raise ASTContractError("AST checkpoint contract mismatch")
    processor = load_processor(directory)
    config = ASTConfig.from_json_file(directory / "config.json")
    if metadata.get("hidden_size") != config.hidden_size:
        raise ASTContractError("AST checkpoint contract mismatch")
    model = ASTMultiTask(ASTModel(config), mode)
    model.load_state_dict(load_file(directory / "model.safetensors"), strict=True)
    return model, processor


In [ ]:
INPUT_ARCHIVES = [{'name': 'v4_train.zip', 'bytes': 3150620485, 'sha256': 'bb253c93b3cc0b129cf8b0661b764b90f6578de50ef8f5eec4180231a67cb29b', 'parts': [{'name': 'v4_train.zip.part000', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part000', 'bytes': 99614720, 'sha256': '3115674afac22a6f70ecf42b907583486084b6c2a35805f9036ba108ec268e39', 'drive_id': '1EuP06PC8sdHaXlvOtWgezr5G9zIjtUMI', 'url': 'https://drive.google.com/file/d/1EuP06PC8sdHaXlvOtWgezr5G9zIjtUMI/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part001', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part001', 'bytes': 99614720, 'sha256': 'a23289a307bb0e5d4873dfacf5e49b6dc7728e5039efbbd24049977f79ac7982', 'drive_id': '1MjebPnkeq8c-LSmnBJkKJwTOKTzZp_Co', 'url': 'https://drive.google.com/file/d/1MjebPnkeq8c-LSmnBJkKJwTOKTzZp_Co/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part002', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part002', 'bytes': 99614720, 'sha256': '6a9c1eb54c3553ac212c501a8f4686a773ad89cccba6fdaf21409bffea85b257', 'drive_id': '1Y2ida30neXlq6VfPwWp-TMYzpms0cwDq', 'url': 'https://drive.google.com/file/d/1Y2ida30neXlq6VfPwWp-TMYzpms0cwDq/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part003', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part003', 'bytes': 99614720, 'sha256': 'a6a6483eeda57442d652d9d449f78d65b07e39cb58c898310e74546910460836', 'drive_id': '1MSvXaiR_qMrt3dbm41r87xmRmaX2TGEk', 'url': 'https://drive.google.com/file/d/1MSvXaiR_qMrt3dbm41r87xmRmaX2TGEk/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part004', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part004', 'bytes': 99614720, 'sha256': '3eb9e98f1f7dca89cb5a3600ae49ab1126345dd7adc3e02d3235dcdb543cd091', 'drive_id': '1cM4oLpvuYR8-aNv1Jz18nRw0vZvWfE9z', 'url': 'https://drive.google.com/file/d/1cM4oLpvuYR8-aNv1Jz18nRw0vZvWfE9z/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part005', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part005', 'bytes': 99614720, 'sha256': 'a1f7f5baeea36f91d92163707e275ee1e0b9f5cd51f6f32802c0e83b5e8a65f1', 'drive_id': '1xufJGo5mKmeFSAXTF9Aha5EZo9R6v6Yt', 'url': 'https://drive.google.com/file/d/1xufJGo5mKmeFSAXTF9Aha5EZo9R6v6Yt/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part006', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part006', 'bytes': 99614720, 'sha256': 'dc8868bb5bea1683eace5ed8d4f0dc3bd48ee359483f2e9cf0dfbcd24275db57', 'drive_id': '1ehZUFKTp66qS-MJPKU2IbL04SET0sRo8', 'url': 'https://drive.google.com/file/d/1ehZUFKTp66qS-MJPKU2IbL04SET0sRo8/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part007', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part007', 'bytes': 99614720, 'sha256': '7ca39dbbb9d7c733dfa8023c03f8c2dbb9f33724c4d0a2a13b8a604948bb9824', 'drive_id': '1t_-EUBbo-aa0kqlIYuCgg-MSc66VDR8d', 'url': 'https://drive.google.com/file/d/1t_-EUBbo-aa0kqlIYuCgg-MSc66VDR8d/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part008', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part008', 'bytes': 99614720, 'sha256': '4c3b273e3fe6c3cce33979d4da800f71801fe1439551ebc7d46231f1e7c33d94', 'drive_id': '1PU6Wl0yE_pzaHbeQadd2QIMm1yM7rxUG', 'url': 'https://drive.google.com/file/d/1PU6Wl0yE_pzaHbeQadd2QIMm1yM7rxUG/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part009', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part009', 'bytes': 99614720, 'sha256': '1b15209238b6481ea385243ffeba43dea17637894323254f225f417545af5308', 'drive_id': '1J8Kjv1cprtmBJHNzsHmSem9__SjxH--3', 'url': 'https://drive.google.com/file/d/1J8Kjv1cprtmBJHNzsHmSem9__SjxH--3/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part010', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part010', 'bytes': 99614720, 'sha256': '3d19705f4ab1fefe140c6693205b0b221fd0b94cb76d62ebf4c7b9b96005bfea', 'drive_id': '1wyeFmV7qvklc7ld44197P2iYQDqv4nrl', 'url': 'https://drive.google.com/file/d/1wyeFmV7qvklc7ld44197P2iYQDqv4nrl/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part011', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part011', 'bytes': 99614720, 'sha256': '6b5a050eb28581e1a92fa6b1dad7ae6fc14491c1eab312bbd5cc55e1cf9b04f4', 'drive_id': '1lEHgvUdVR-9ynT2rBm4a30eOgI6lzwsX', 'url': 'https://drive.google.com/file/d/1lEHgvUdVR-9ynT2rBm4a30eOgI6lzwsX/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part012', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part012', 'bytes': 99614720, 'sha256': '8716e2febeddc8a5c4d398e01272260f6dd5ed1fb2d02d0e12db8c67c3d89c5b', 'drive_id': '12O0awlYjMiC0uBkS_yYJDX6NwpS1nn05', 'url': 'https://drive.google.com/file/d/12O0awlYjMiC0uBkS_yYJDX6NwpS1nn05/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part013', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part013', 'bytes': 99614720, 'sha256': '81c8a6fef01ed4eed420cf29fa628783486cd1496748e57cd62aea163394febd', 'drive_id': '1xW8ya0UO9zS7KBORHu2BhoMHM6v5oPaS', 'url': 'https://drive.google.com/file/d/1xW8ya0UO9zS7KBORHu2BhoMHM6v5oPaS/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part014', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part014', 'bytes': 99614720, 'sha256': '271d8790b63e3769bc3e3ed6010d7b40602dfb96d9a824e9a9b795f4e64bc77d', 'drive_id': '1W_fuwFfw68RPvkmUCopbkqFILYoqzlNg', 'url': 'https://drive.google.com/file/d/1W_fuwFfw68RPvkmUCopbkqFILYoqzlNg/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part015', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part015', 'bytes': 99614720, 'sha256': 'a630c1bef390f3bb8ee4d2600ba4c2abf65ebbbad8ffe7b3825c4252f833d1a1', 'drive_id': '1haiLH0O1c-TDF1v0XKbel2Dcu3cc720N', 'url': 'https://drive.google.com/file/d/1haiLH0O1c-TDF1v0XKbel2Dcu3cc720N/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part016', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part016', 'bytes': 99614720, 'sha256': '89208dceae8948f6677e5468e5efd9dccc596a773a5b093383967b6355e9c5ea', 'drive_id': '1C2kHDjhB92QPJxQtNA3Wdebvz7sJM2eb', 'url': 'https://drive.google.com/file/d/1C2kHDjhB92QPJxQtNA3Wdebvz7sJM2eb/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part017', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part017', 'bytes': 99614720, 'sha256': '3458326fba9c4cc5e47c6149408a1c0840585692be465704faa5842ce02c762b', 'drive_id': '1iFoGfgyf1YdxlgekgP0a-DmaSLmFxwcy', 'url': 'https://drive.google.com/file/d/1iFoGfgyf1YdxlgekgP0a-DmaSLmFxwcy/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part018', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part018', 'bytes': 99614720, 'sha256': '84ef668eae8e620c8482a70108b76083d10fbfc17b17c23acdc49b082d735302', 'drive_id': '1fyAyFWAPtIA4O8rGMKHclKLQRgBm00H5', 'url': 'https://drive.google.com/file/d/1fyAyFWAPtIA4O8rGMKHclKLQRgBm00H5/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part019', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part019', 'bytes': 99614720, 'sha256': '20340d4e08f3f4ef1b438273357600a435423b8694bd7f9147ab8034fe5ac737', 'drive_id': '1mXqlFR0uL3KFVVOdoTJrziHJFJIMmy-A', 'url': 'https://drive.google.com/file/d/1mXqlFR0uL3KFVVOdoTJrziHJFJIMmy-A/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part020', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part020', 'bytes': 99614720, 'sha256': '3a561e5172641dd98c708cb70ebb7d58027ba73bfd66ba15bed20073c7302042', 'drive_id': '1onAyxKaxFsVKiWX51Z_JNWsSpCi3-2HO', 'url': 'https://drive.google.com/file/d/1onAyxKaxFsVKiWX51Z_JNWsSpCi3-2HO/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part021', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part021', 'bytes': 99614720, 'sha256': 'bdb0271235315c1a88ac09625635ff919128356e76fe4515ed33a5b5af263373', 'drive_id': '111eaO10UoWZcUU2jqYAhFBx0Y3VHc4yN', 'url': 'https://drive.google.com/file/d/111eaO10UoWZcUU2jqYAhFBx0Y3VHc4yN/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part022', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part022', 'bytes': 99614720, 'sha256': 'c393345f714cb4ac631ca42e4f05ed07ddc0d846b182073a2c29a0344d8b14eb', 'drive_id': '1Djxad7z-I-6BioHzwFDfufL8zFKziPaO', 'url': 'https://drive.google.com/file/d/1Djxad7z-I-6BioHzwFDfufL8zFKziPaO/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part023', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part023', 'bytes': 99614720, 'sha256': '89132b498eaaba4923e215bd7399cdc429de334409f0e8170f884f53755bd23b', 'drive_id': '1QIETx7ktcvQ2Vx4fTRltAbr2ytM3sb4C', 'url': 'https://drive.google.com/file/d/1QIETx7ktcvQ2Vx4fTRltAbr2ytM3sb4C/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part024', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part024', 'bytes': 99614720, 'sha256': '65f6a64d52df62db31e5dd796c05efab9eb2a634a4dcde7536551a040a4121ef', 'drive_id': '1oTLsL5nQiWNzUfrn_Dpa03H2caV7ym9i', 'url': 'https://drive.google.com/file/d/1oTLsL5nQiWNzUfrn_Dpa03H2caV7ym9i/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part025', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part025', 'bytes': 99614720, 'sha256': '20631d301cc47d8cad698dfa8b2e0436971151af901478279b44abf4e0d10315', 'drive_id': '1OcgbQsaVsr5b1wSrrAp6pT6FnpbONzZl', 'url': 'https://drive.google.com/file/d/1OcgbQsaVsr5b1wSrrAp6pT6FnpbONzZl/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part026', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part026', 'bytes': 99614720, 'sha256': '116a7f4066af76cff9fbf5b3045cb622e37b36a50927c372130a50f99da168d7', 'drive_id': '1_zI6HVmREUBU9JI229shoGymLYPVF7iR', 'url': 'https://drive.google.com/file/d/1_zI6HVmREUBU9JI229shoGymLYPVF7iR/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part027', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part027', 'bytes': 99614720, 'sha256': 'aa463411a1583518c8abdbf083a2e84ebf22205bf1c0688c60098be6e5699f00', 'drive_id': '1Vx5pS8HMdbAxViE0H5_kvHsC9Zwy4jy-', 'url': 'https://drive.google.com/file/d/1Vx5pS8HMdbAxViE0H5_kvHsC9Zwy4jy-/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part028', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part028', 'bytes': 99614720, 'sha256': '42725274198cab7f673e073521d372e5616c2a450e9ed60b5417e03476227eb7', 'drive_id': '16qmzYHny-LWm40-Mz973h6BfmCZmfROc', 'url': 'https://drive.google.com/file/d/16qmzYHny-LWm40-Mz973h6BfmCZmfROc/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part029', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part029', 'bytes': 99614720, 'sha256': '8bfd0ec341b13a2c69ff8a93865a15f4e1c2ad566852c65b8aacc8fa753ccfb1', 'drive_id': '1ctRZXqC5kZjVc9bp_RkpiTHJyBvhfZWn', 'url': 'https://drive.google.com/file/d/1ctRZXqC5kZjVc9bp_RkpiTHJyBvhfZWn/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part030', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part030', 'bytes': 99614720, 'sha256': 'f924f479ae00badbd83cb418af8e7a6b99946608ec2f199c112602dd1f842146', 'drive_id': '12twFDp4H4vhYmmn8PeeLJd-DX7a5nYb8', 'url': 'https://drive.google.com/file/d/12twFDp4H4vhYmmn8PeeLJd-DX7a5nYb8/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part031', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part031', 'bytes': 62564165, 'sha256': '7fe31a9a65f460f84431375557772010eb80f8dccd2b2cc9ba6a384e43d567d4', 'drive_id': '1VVEwZAIUxXa0stmxRy_DhDUCQR0Y5ln5', 'url': 'https://drive.google.com/file/d/1VVEwZAIUxXa0stmxRy_DhDUCQR0Y5ln5/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}]}, {'name': 'v4_dev.zip', 'bytes': 255421320, 'sha256': 'b1e0c5771811deed0bb2e7bc99644b24cd4cb827e2166b9f56ff2d6774e8264f', 'parts': [{'name': 'v4_dev.zip.part000', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_dev.zip.part000', 'bytes': 99614720, 'sha256': '5d4cebfc887e9e45759495d9de9bd49f2d69fa607ea11932c4961b7b6db585a5', 'drive_id': '18a68J3eQJLUcMyuAV1_SGcXZHBtfIeB2', 'url': 'https://drive.google.com/file/d/18a68J3eQJLUcMyuAV1_SGcXZHBtfIeB2/view?usp=drivesdk', 'archive': 'v4_dev.zip', 'drive_metadata_verified': True}, {'name': 'v4_dev.zip.part001', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_dev.zip.part001', 'bytes': 99614720, 'sha256': '6213d3d29b5597e8d65d92e835bc5b8d14a2a75d8e8f496bc26cf7bf1320bd03', 'drive_id': '1NhxvwEl2ecJhGcAW60QmQYIsNwSGOzPd', 'url': 'https://drive.google.com/file/d/1NhxvwEl2ecJhGcAW60QmQYIsNwSGOzPd/view?usp=drivesdk', 'archive': 'v4_dev.zip', 'drive_metadata_verified': True}, {'name': 'v4_dev.zip.part002', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_dev.zip.part002', 'bytes': 56191880, 'sha256': '2c836e31ffbe1269df390be63239dc6478f58af4ca690b76438c83babee4d960', 'drive_id': '1tyr7gQaIqufXNYFjeShK_SNWETg2a5np', 'url': 'https://drive.google.com/file/d/1tyr7gQaIqufXNYFjeShK_SNWETg2a5np/view?usp=drivesdk', 'archive': 'v4_dev.zip', 'drive_metadata_verified': True}]}, {'name': 'v4_sealed.zip', 'bytes': 322426108, 'sha256': '5d4a678d6fd299796fb454bf9b42aa150d5ce10694c3a245a1a4657fc39d5ccc', 'parts': [{'name': 'v4_sealed.zip.part000', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_sealed.zip.part000', 'bytes': 99614720, 'sha256': '3ba4be13cc2298f45367f95c5b1b39297b7b2ae0603acb2e8d9a8394d612a97b', 'drive_id': '1c7hgud7i1hJn1KwBxmgbwcRoDARufydx', 'url': 'https://drive.google.com/file/d/1c7hgud7i1hJn1KwBxmgbwcRoDARufydx/view?usp=drivesdk', 'archive': 'v4_sealed.zip', 'drive_metadata_verified': True}, {'name': 'v4_sealed.zip.part001', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_sealed.zip.part001', 'bytes': 99614720, 'sha256': '449e1c7d3b3ef0de31470154cd38ea5869dd44c6faddb393ea5d1fa4c885ea47', 'drive_id': '1iiFVgntk55A22B9hfsDlMEa-6Hn2a1WL', 'url': 'https://drive.google.com/file/d/1iiFVgntk55A22B9hfsDlMEa-6Hn2a1WL/view?usp=drivesdk', 'archive': 'v4_sealed.zip', 'drive_metadata_verified': True}, {'name': 'v4_sealed.zip.part002', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_sealed.zip.part002', 'bytes': 99614720, 'sha256': 'ed6112a0f30df1780bc1210d43112c18d40ff6a3ce028bd259a6b2d4ddc608fa', 'drive_id': '1IfsV5ZFK8cexNjRMKAbBWcG7z6vmSjsz', 'url': 'https://drive.google.com/file/d/1IfsV5ZFK8cexNjRMKAbBWcG7z6vmSjsz/view?usp=drivesdk', 'archive': 'v4_sealed.zip', 'drive_metadata_verified': True}, {'name': 'v4_sealed.zip.part003', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_sealed.zip.part003', 'bytes': 23581948, 'sha256': '640c52fb871f5684be81724306ff6a94a583bec790c6a18707ff549423d8225d', 'drive_id': '1LijiiHrod7c6es8SovjPgx4JUTf4UjZg', 'url': 'https://drive.google.com/file/d/1LijiiHrod7c6es8SovjPgx4JUTf4UjZg/view?usp=drivesdk', 'archive': 'v4_sealed.zip', 'drive_metadata_verified': True}]}, {'name': 'v4_metadata.zip', 'bytes': 659843, 'sha256': '89bc9340bb0860299eabb7934488aca3c1199ad2043a783f5ede71f9e7d50d30', 'parts': [{'name': 'v4_metadata.zip.part000', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_metadata.zip.part000', 'bytes': 659843, 'sha256': '89bc9340bb0860299eabb7934488aca3c1199ad2043a783f5ede71f9e7d50d30', 'drive_id': '1C5g9GYw5x1btyZr0PpAvtWOi3tL7fRL5', 'url': 'https://drive.google.com/file/d/1C5g9GYw5x1btyZr0PpAvtWOi3tL7fRL5/view?usp=drivesdk', 'archive': 'v4_metadata.zip', 'drive_metadata_verified': True}]}, {'sha256': '54c227b873cc3c424579fa06e8a06c442b91663de5b740b182f7cac30a3fac75', 'bytes': 346433653, 'model_revision': 'f826b80d28226b62986cc218e5cec390b1096902', 'name': 'v2_ast_assets.zip', 'parts': [{'name': 'v2_ast_assets.zip.part00', 'bytes': 83886080, 'sha256': 'f4f095b3c6062b341f6f2554c210254170da87281ee1eb547b4b6a09940764dc'}, {'name': 'v2_ast_assets.zip.part01', 'bytes': 83886080, 'sha256': 'd277ef91dcef6fe298c48ce23bbb3289b7b49e3748bd5fae2cf614f91c0c2e4d'}, {'name': 'v2_ast_assets.zip.part02', 'bytes': 83886080, 'sha256': 'df38da76de6f510582055ba33225502e7b9fa1747f487742f3dd762b12e629fb'}, {'name': 'v2_ast_assets.zip.part03', 'bytes': 83886080, 'sha256': 'b575be13671d5c3ffbc0ba74cf8a4c701c91d9270c0bffd473a7a105a623f96a'}, {'name': 'v2_ast_assets.zip.part04', 'bytes': 10889333, 'sha256': 'f2ade7b797f0ad384c0632b3d0388d4df01a9e70db6465325ac4bf9e8c2d51c3'}]}, {'name': 'recovery_sonics_assets.zip', 'bytes': 69351688, 'sha256': 'fe00b65ef86578e35d1cfb4beefd1a71f4c686b29a5618cc4f85ccb8cbcf521c', 'drive_id': '1Qk4tAPcVZmfIrRZEVi74UtoQJz-fao_l', 'parts': [{'name': 'recovery_sonics_assets.zip', 'bytes': 69351688, 'sha256': 'fe00b65ef86578e35d1cfb4beefd1a71f4c686b29a5618cc4f85ccb8cbcf521c', 'drive_id': '1Qk4tAPcVZmfIrRZEVi74UtoQJz-fao_l'}]}]

## Results

No experimental output is embedded initially. During a successful run, this notebook renders a plot only from the runner-written `epoch_metrics.csv`; its Drive receipt records every displayed input.

In [ ]:
def safe_extract(archive, destination):
    with zipfile.ZipFile(archive) as packed:
        members = packed.infolist()
        if any(Path(member.filename).is_absolute() or '..' in Path(member.filename).parts for member in members):
            raise ValueError(f'Unsafe archive member in {archive.name}')
        packed.extractall(destination)

def capture_executed_notebook():
    try:
        from google.colab import _message
        reply = _message.blocking_request('get_ipynb', timeout_sec=30)
        payload = reply['ipynb']
        if isinstance(payload, str):
            payload = json.loads(payload)
        destination = RUN / 'executed.ipynb'
        destination.write_text(json.dumps(payload), encoding='utf-8')
        write_json(RUN / 'executed_notebook_capture.json', {'status': 'captured', 'capture_stage': 'before_runtime_unassign', 'current_cell_may_be_incomplete': True, 'sha256': sha256(destination)})
    except Exception:
        (RUN / 'executed_notebook_capture_error.txt').write_text(traceback.format_exc(), encoding='utf-8')
        write_json(RUN / 'executed_notebook_capture.json', {'status': 'not_captured'})

def record_artifacts():
    names = ('trials.jsonl', 'epoch_metrics.csv', 'exposures.json', 'selection.json', 'recipe_lock.json', 'refit_manifest.csv', 'final_weight_audit.json', 'weight_lock.json', 'sealed_opened.json', 'failure.txt', 'experiment.log')
    receipt = {}
    for name in names:
        path = RUN / name
        if path.is_file():
            receipt[name] = {'bytes': path.stat().st_size, 'sha256': sha256(path)}
    write_json(RUN / 'drive_receipt.json', receipt)

def plot_results():
    metrics_path = RUN / 'epoch_metrics.csv'
    if not metrics_path.is_file():
        write_json(RUN / 'plot_status.json', {'status': 'not_created', 'reason': 'no executed epoch_metrics.csv'})
        return
    rows = list(csv.DictReader(metrics_path.open(encoding='utf-8')))
    if not rows:
        write_json(RUN / 'plot_status.json', {'status': 'not_created', 'reason': 'empty epoch_metrics.csv'})
        return
    x_name = next((name for name in ('epoch', 'step') if name in rows[0]), None)
    group_name = next((name for name in ('pipeline', 'trial', 'name') if name in rows[0]), None)
    numeric = []
    for name in rows[0]:
        try:
            [float(row[name]) for row in rows if row.get(name, '') != '']
        except ValueError:
            continue
        numeric.append(name)
    metric_name = next((name for name in ('rank_score', 'dev_score', 'proxy_score', 'loss', 'dev_loss') if name in numeric), next((name for name in numeric if name != x_name), None))
    if x_name is None or group_name is None or metric_name is None:
        write_json(RUN / 'plot_status.json', {'status': 'not_created', 'reason': 'missing plotted metric columns', 'columns': list(rows[0])})
        return
    import matplotlib.pyplot as plt
    grouped = {}
    for row in rows:
        if row.get(x_name, '') and row.get(metric_name, ''):
            grouped.setdefault(row[group_name], []).append((float(row[x_name]), float(row[metric_name])))
    if not grouped:
        write_json(RUN / 'plot_status.json', {'status': 'not_created', 'reason': 'no numeric plotted rows'})
        return
    figure, axis = plt.subplots(figsize=(9, 5))
    for label, values in sorted(grouped.items()):
        values.sort()
        axis.plot(*zip(*values), marker='o', label=label)
    axis.set(title=f'Executed V4 {metric_name}', xlabel=x_name, ylabel=metric_name)
    axis.grid(alpha=.25)
    axis.legend()
    figure.tight_layout()
    output = RUN / 'epoch_metric.png'
    figure.savefig(output, dpi=160)
    plt.show()
    write_json(RUN / 'plot_status.json', {'status': 'created', 'path': output.name, 'sha256': sha256(output)})

process = None
try:
    (RUN / 'code').mkdir(exist_ok=True)
    for source in WORK.glob('*.py'):
        shutil.copy2(source, RUN / 'code' / source.name)
    write_json(RUN / 'source_hashes.json', {path.name: sha256(path) for path in sorted((RUN / 'code').glob('*.py'))})
    write_json(RUN / 'input_archives.json', INPUT_ARCHIVES)
    packages = ['demucs==4.0.1', 'panns-inference==0.1.1', 'transformers==4.57.6', 'librosa==0.10.2.post1', 'diffq==0.2.4', 'einops==0.8.1', 'timm==1.0.15', 'scikit-learn==1.8.0', 'soundfile==0.12.1', 'imageio-ffmpeg', 'safetensors', 'fvcore==0.1.5.post20221221']
    with (RUN / 'install.log').open('w', encoding='utf-8') as log:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], stdout=log,
                       stderr=subprocess.STDOUT, check=True, timeout=600)
    import torch
    if not torch.cuda.is_available() or 'A100' not in torch.cuda.get_device_name():
        raise RuntimeError('Requested A100 runtime is unavailable')
    version_names = ('torch', 'torchaudio', 'torchvision', 'transformers', 'fvcore', 'timm', 'librosa')
    write_json(RUN / 'environment.json', {'python': sys.version, 'gpu': torch.cuda.get_device_name(), 'packages': {name: importlib.metadata.version(name) for name in version_names}})
    for archive in INPUT_ARCHIVES:
        destination = WORK / archive['name']
        reusable = destination.is_file() and destination.stat().st_size == archive['bytes'] and sha256(destination) == archive['sha256']
        if not reusable:
            with destination.open('wb') as output:
                for part in archive['parts']:
                    source = DRIVE / part['name']
                    if source.stat().st_size != part['bytes'] or sha256(source) != part['sha256']:
                        raise ValueError(f'Drive part receipt mismatch: {source.name}')
                    with source.open('rb') as incoming:
                        shutil.copyfileobj(incoming, output, 8 * 1024**2)
        if destination.stat().st_size != archive['bytes'] or sha256(destination) != archive['sha256']:
            raise ValueError(f'Archive receipt mismatch: {destination.name}')
        safe_extract(destination, WORK)
        budget_event('archive_verified', archive=destination.name, reused_local_archive=reusable)
    if not all(path.is_file() for path in (WORK / 'v4_validation/train.csv', WORK / 'v4_validation/dev.csv', WORK / 'v4_validation/sealed.csv', WORK / 'sonics_assets/config.json', WORK / 'sonics_assets/pytorch_model.bin', WORK / 'ast/config.json', WORK / 'ast/preprocessor_config.json', WORK / 'ast/model.safetensors')): raise FileNotFoundError('Verified archives did not produce the required V4 roots')
    shutil.copy2(DRIVE / 'open.zip', WORK / 'open.zip')
    official = WORK / 'official'
    safe_extract(WORK / 'open.zip', official)
    original = next(official.rglob('baseline_submit.zip'))
    if sha256(original) != 'cc86f143915c95fd958a0066cbf14b0db9f4c41fd213310c3650b9fe870d0c70':
        raise ValueError('Official baseline archive hash changed')
    baseline = WORK / 'baseline'; safe_extract(original, baseline)
    import demucs, diffq, panns_inference
    if not (baseline / 'script.py').is_file() or not (baseline / 'model' / 'SHA256SUMS.txt').is_file():
        raise FileNotFoundError('Baseline script or signed model inventory is missing')
    print('CPU_PREFLIGHT_OK', importlib.metadata.version('demucs'), flush=True)
    command = [sys.executable, '-u', 'train_v4.py', '--data', str(WORK / 'v4_validation'), '--baseline', str(baseline), '--sonics', str(WORK / 'sonics_assets'), '--ast', str(WORK / 'ast'), '--output', str(RUN), '--deadline', str(SAFE_DEADLINE)]
    environment = dict(os.environ, PYTHONPATH=str(WORK / 'sonics_vendor'), HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1')
    budget_event('runner_started', command=command)
    with (RUN / 'experiment.log').open('w', buffering=1, encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=WORK, stdout=log, stderr=subprocess.STDOUT, env=environment)
        while process.poll() is None:
            if time.time() >= SAFE_DEADLINE - 60:
                budget_event('runner_terminated_for_safe_shutdown')
                process.terminate()
                process.wait(timeout=30)
                raise TimeoutError('V4 runner reached the safe allocation boundary')
            time.sleep(15)
            lines = (RUN / 'experiment.log').read_text(encoding='utf-8', errors='replace').splitlines()
            print(lines[-1][:1000] if lines else 'Preparing V4 experiment...', flush=True)
    if process.returncode:
        raise RuntimeError(f'V4 runner exited with {process.returncode}')
    budget_event('runner_finished')
    plot_results()
    print('V4_SAVED', RUN, flush=True)
except Exception:
    (RUN / 'failure.txt').write_text(traceback.format_exc(), encoding='utf-8')
    budget_event('notebook_failure')
    print(traceback.format_exc(), flush=True)
    raise
finally:
    capture_executed_notebook()
    record_artifacts()
    write_json(RUN / 'release_requested.json', {'unix': time.time(), 'elapsed_seconds': time.time() - SESSION_START})
    watchdog.cancel()
    runtime.unassign()

## Takeaways

Initial state: no V4 result, no selected candidate, and no DACON submission. Read `selection.json`, the CSVs, budget events, and the executed-notebook capture after a run before making any performance claim.